# Spark Interview Prep — Medium Level

**Audience:** Engineers with 6–12 years of experience preparing for Staff/Principal Spark interviews.

**Topics Covered:**
1. Broadcast Join
2. Shuffle Optimization
3. Window Functions
4. Bucketing
5. Data Skew Detection & Salting
6. Adaptive Query Execution (AQE)

**Dataset:** NYC Cab Registry (`Cabs.csv`, ~8 970 rows) + synthetically generated trip/fare data.

---
> **How to use this notebook:** Work through each topic sequentially. Run the *bad* cell first, observe the plan/metrics, then run the *optimized* cell and compare. Answer the interview questions before reading the discussion sections.

In [ ]:
import os, sys
os.environ["PYSPARK_PYTHON"] = sys.executable

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
from pyspark.sql.window import Window
import time

active = SparkSession.getActiveSession()
if active:
    active.stop()

spark = SparkSession.builder \
    .appName("SparkInterviewPrep-Medium") \
    .config("spark.sql.adaptive.enabled", "true") \
    .config("spark.sql.shuffle.partitions", "8") \
    .config("spark.sql.autoBroadcastJoinThreshold", "-1") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")
CABS_PATH = r"C:\Users\PS\Documents\Python-Exp\RawData\Cabs.csv"
print("Spark version:", spark.version)

## Topic 1 — Broadcast Join

### Business Scenario
The NYC TLC analytics team processes **500 million trip records per day**. Each trip must be enriched with cab registry metadata (license type, vehicle year, wheelchair accessibility) to power downstream reporting dashboards. The cab registry (`Cabs.csv`) is a **slowly-changing dimension** with ~8 970 rows — it changes at most once a day.

### Dataset Description
| Table | Rows | Size (est.) | Join Key |
|-------|------|-------------|----------|
| `trips` (synthetic) | 500 000 (demo) / 500 M (prod) | ~40 GB | `cab_number` |
| `cabs` (Cabs.csv) | 8 970 | ~2 MB | `CabNumber` |

### Schema
```
trips:  cab_number STRING, pickup_ts TIMESTAMP, fare DOUBLE, distance DOUBLE, borough STRING
cabs:   CabNumber STRING, Name STRING, LicenseType STRING, Active STRING,
        VehicleYear INT, WheelchairAccessible STRING
```

### Why this matters in interviews
Broadcast join is the **single highest-ROI optimization** available in Spark. Misusing SortMergeJoin when one table fits in driver/executor memory is a textbook seniority signal. Interviewers will probe: threshold tuning, driver OOM risk, and when broadcast *hurts*.

In [ ]:
# ── Sample Data Generation ────────────────────────────────────────────────────
# Load the real cab registry (small dimension table)
cabs_df = spark.read.option("header", True).option("inferSchema", True).csv(CABS_PATH)

# Keep only the columns we need
cabs_df = cabs_df.select(
    F.col("CabNumber"),
    F.col("Name"),
    F.col("LicenseType"),
    F.col("Active"),
    F.col("VehicleYear"),
    F.col("WheelchairAccessible")
).cache()  # Dimension tables are good cache candidates

print(f"Cab registry rows: {cabs_df.count():,}")

# Extract distinct cab numbers to use as a realistic join key pool
cab_numbers = [r.CabNumber for r in cabs_df.select("CabNumber").collect()]
print(f"Distinct cab numbers in registry: {len(cab_numbers):,}")

# Generate synthetic trips (500_000 rows for local demo; comment says 500M for prod)
boroughs = ["Manhattan", "Brooklyn", "Queens", "Bronx", "Staten Island"]

trips_df = (
    spark.range(500_000)  # 500 000 rows locally; scale to 500_000_000 in prod
    .withColumn("cab_number",
        F.element_at(
            F.array([F.lit(c) for c in cab_numbers[:50]]),  # use first 50 cabs for demo
            (F.col("id") % 50 + 1).cast("int")
        )
    )
    .withColumn("pickup_ts",
        F.to_timestamp(
            F.date_add(F.lit("2024-01-01"), (F.col("id") % 365).cast("int")), "yyyy-MM-dd"
        )
    )
    .withColumn("fare",     F.round(F.rand(seed=42) * 80 + 5, 2))
    .withColumn("distance", F.round(F.rand(seed=7)  * 20 + 0.5, 2))
    .withColumn("borough",
        F.element_at(
            F.array([F.lit(b) for b in boroughs]),
            (F.col("id") % 5 + 1).cast("int")
        )
    )
    .drop("id")
)

print(f"Synthetic trip rows: {trips_df.count():,}")
trips_df.printSchema()

### Problem Statement
Enrich every trip row with the cab's `LicenseType` and `VehicleYear` from the cab registry. The naive implementation forces Spark to **sort and shuffle both sides** of the join (SortMergeJoin), exchanging hundreds of GBs of network traffic — even though the right-hand table is 2 MB.

In [ ]:
# ── BAD CODE: SortMergeJoin (no broadcast hint, threshold disabled) ───────────
# autoBroadcastJoinThreshold is -1 in our session, so Spark will NEVER auto-broadcast.
# The planner falls back to SortMergeJoin: sort both sides, shuffle both sides.

print("=" * 60)
print("BAD: SortMergeJoin plan")
print("=" * 60)

bad_join = trips_df.join(cabs_df, trips_df.cab_number == cabs_df.CabNumber, "left")

# Show the physical plan — look for:
#   Exchange hashpartitioning  ← shuffle on BOTH sides
#   Sort                        ← sort on both sides before merge
#   SortMergeJoin               ← the join operator itself
bad_join.explain(mode="formatted")

### Interview Questions — Broadcast Join

Answer these before reading the discussion sections below.

1. Spark uses SortMergeJoin by default for large tables. Walk me through every stage Spark creates for a SortMergeJoin between a 500 GB fact table and a 2 MB dimension table. How many shuffles happen?

2. What is `spark.sql.autoBroadcastJoinThreshold` and how does Spark measure the size of a table against this threshold? Is it pre-compression or post-compression size?

3. You add `F.broadcast(small_df)` but the physical plan still shows `SortMergeJoin`. Name three reasons this can happen.

4. At what table size does broadcast become counter-productive? Consider both driver memory (collection cost) and executor memory (replication across all executors on a 500-node cluster).

5. You have a join between two 10 GB tables. Neither fits in broadcast threshold. What join strategies does Spark consider, and what are the trade-offs between SortMergeJoin and ShuffledHashJoin?

6. A broadcast join is succeeding in development (50 executors) but causing OOM in production (500 executors). Walk through why and how you fix it.

7. How does `spark.sql.broadcastTimeout` interact with driver-side collection? What happens if the driver takes longer than this timeout to serialize and distribute the broadcast table?

8. You have a stream-static join in Spark Structured Streaming. Can you use broadcast on the static side? What are the limitations?

### Expected Logical Plan Discussion

**Unresolved Logical Plan (analyst view):**
```
Project [cab_number, pickup_ts, fare, distance, borough, Name, LicenseType, VehicleYear]
+- Join LeftOuter, (cab_number = CabNumber)
   :- Range (0, 500000, step=1, splits=8)   ← trips side
   +- Relation [CabNumber, Name, ...]  CSVFileFormat   ← cabs side
```

**After Analysis (resolved references, type coercion applied):**
The optimizer walks both sides, infers statistics from CSV headers and file sizes, and checks `autoBroadcastJoinThreshold`. With the threshold at -1, no broadcast is attempted regardless of the cabs table size.

**After Optimization (Catalyst rules applied):**
- `EliminateSubqueryAliases` — removes redundant aliases
- `PushDownPredicates` — any WHERE clauses pushed below the join
- `ColumnPruning` — only selected columns retained from cabs
- The join remains `Join LeftOuter` — no strategy selection yet (that happens in physical planning)

**Key insight:** Catalyst's logical optimizer does *not* choose join strategies. That is the physical planner's job. The `JoinSelection` physical planning rule evaluates broadcast eligibility.

### Expected Physical Plan Discussion

**Without broadcast hint (bad path):**
```
AdaptiveSparkPlan (isFinalPlan=false)
+- SortMergeJoin [cab_number], [CabNumber], LeftOuter
   :- Sort [cab_number ASC NULLS FIRST], false
   :  +- Exchange hashpartitioning(cab_number, 8)       ← SHUFFLE #1 (trips)
   :     +- Project [cab_number, pickup_ts, fare, ...]
   :        +- Range (0, 500000, ...)
   +- Sort [CabNumber ASC NULLS FIRST], false
      +- Exchange hashpartitioning(CabNumber, 8)         ← SHUFFLE #2 (cabs)
         +- Project [CabNumber, Name, LicenseType, ...]
            +- FileScan csv [...]
```
**Cost:** 2 shuffles + 2 sorts. On 500M rows × 40 GB, Shuffle #1 alone writes 40 GB across the network.

**With broadcast hint (optimized path):**
```
AdaptiveSparkPlan (isFinalPlan=false)
+- BroadcastHashJoin [cab_number], [CabNumber], LeftOuter, BuildRight
   :- Project [cab_number, pickup_ts, fare, ...]
   :  +- Range (0, 500000, ...)
   +- BroadcastExchange HashedRelationBroadcastMode(List(CabNumber))
      +- Project [CabNumber, Name, LicenseType, ...]
         +- FileScan csv [...]
```
**Cost:** 0 shuffles. The cabs table (2 MB) is collected to the driver, serialized as a `HashedRelation`, and pushed to every executor via `SparkContext.broadcast()`. Each executor probes the in-memory hash map independently.

**Node names to cite in interviews:**
- `BroadcastExchange` — driver collects and broadcasts the small table
- `BroadcastHashJoin` — executor-side hash probe (no sort needed)
- `HashedRelationBroadcastMode` — the hash map format used for the broadcast variable

### Spark UI Investigation Guide — Broadcast Join

**Where to look:**

| UI Tab | What to check | Red flag |
|--------|--------------|----------|
| SQL / DataFrame | Physical plan graph | `Exchange` nodes on both join inputs = SortMergeJoin |
| Stages | Shuffle Read/Write bytes | Non-zero shuffle bytes = data moved over network |
| Executors | Peak memory per executor | Unusually high = broadcast var taking executor heap |
| SQL / DataFrame | `BroadcastExchange` node timing | > 5 min = hit `broadcastTimeout`, job will fail |

**Key metrics to compare (bad vs good):**
```
Metric                   SortMergeJoin     BroadcastHashJoin
─────────────────────────────────────────────────────────────
Shuffle Write (trips)    ~40 GB            0 B
Shuffle Read  (all)      ~40 GB            0 B
Stages                   3 (map+shuffle×2) 1 (map only)
Tasks (shuffle stage)    200+              0
Wall clock time (500M)   ~18 min           ~4 min
```

**How to find broadcast size:** In the SQL plan graph, click the `BroadcastExchange` node → look for the metric `data size` — this is the serialized size broadcast to each executor.

In [ ]:
# ── OPTIMIZATION EXERCISE: Fill in the TODOs ─────────────────────────────────
# TODO 1: Apply the correct hint function to force BroadcastHashJoin on cabs_df
# TODO 2: Verify the plan contains 'BroadcastHashJoin' and NOT 'SortMergeJoin'
# TODO 3: Select only the columns needed from the enriched result
# TODO 4: Write the result to Parquet with snappy compression

# Your code here:
# optimized_join = trips_df.join(TODO, trips_df.cab_number == cabs_df.CabNumber, "left")
# optimized_join.explain(mode="formatted")
# optimized_join.select(...).write.mode("overwrite").parquet("/tmp/enriched_trips")

In [ ]:
# ── PERFORMANCE BENCHMARKING ──────────────────────────────────────────────────
import time

def bench(label, df):
    t0 = time.time()
    n = df.count()
    elapsed = time.time() - t0
    print(f"{label:40s}  rows={n:>9,}  time={elapsed:6.2f}s")
    return elapsed

print("─" * 70)

# Bad: SortMergeJoin (AQE may upgrade to BroadcastHashJoin at runtime;
#      force SMJ by using repartition to prevent AQE broadcast)
bad_join_forced = (
    trips_df.repartition(8, F.col("cab_number"))
    .join(
        cabs_df.repartition(8, F.col("CabNumber")),
        trips_df.cab_number == cabs_df.CabNumber,
        "left"
    )
)
t_bad = bench("SortMergeJoin (manual repartition)", bad_join_forced)

# Good: explicit broadcast
good_join = trips_df.join(
    F.broadcast(cabs_df),
    trips_df.cab_number == cabs_df.CabNumber,
    "left"
)
t_good = bench("BroadcastHashJoin (explicit hint) ", good_join)

print("─" * 70)
print(f"Speedup: {t_bad / t_good:.1f}x  (grows dramatically with real 500M-row data)")

### Production Best Practices — Broadcast Join

1. **Always use explicit `F.broadcast()` hints** rather than relying on `autoBroadcastJoinThreshold`. Thresholds are fragile: CSV files have no stats, Parquet stats can be stale, and AQE can override them at runtime in surprising ways.

2. **Set a safe threshold** — `spark.sql.autoBroadcastJoinThreshold = 10mb` (default) is reasonable. Go no higher than 100 MB unless you've profiled executor heap usage. Each executor holds a full copy.

3. **Cache the broadcast table upstream** if it's used in multiple joins in the same job — the CSV read doesn't need to repeat.

4. **Avoid broadcasting mutable DataFrames** — if the dimension table is updated mid-job, you'll join against a stale snapshot. Explicit re-reads with versioned paths are safer.

5. **Monitor `BroadcastExchange` timing** in Spark UI. If the broadcast collection takes > `spark.sql.broadcastTimeout` (default 300 s), the job fails with `SparkException: Could not execute broadcast in 300 secs`. Tune timeout or reduce table size.

6. **In Structured Streaming**, broadcast joins on the static side work but the broadcast variable is **not refreshed** between micro-batches unless you restart the stream or use `foreachBatch` with explicit re-broadcast logic.

### Common Follow-up Questions — Broadcast Join

**Q: What is `BuildSide` in `BroadcastHashJoin [BuildRight]`?**
A: Spark builds the hash map from the `BuildRight` (broadcast) side and probes it with rows from the left side. For an inner join, Spark may choose either side. For a left outer join, the right side must be the build side.

**Q: Can you broadcast a DataFrame that's the result of an aggregation?**
A: Yes — and this is a common pattern. Aggregate first (reducing size), then broadcast the result. Example: daily totals per cab (~8970 rows) can be broadcast to enrich a 500M-row stream.

**Q: How does AQE interact with broadcast thresholds?**
A: With AQE enabled, Spark can *upgrade* a SortMergeJoin to BroadcastHashJoin at runtime if the runtime statistics show one side is below threshold — even if the planner chose SMJ at compile time. This is the `spark.sql.adaptive.autoBroadcastJoinThreshold` knob (separate from the static threshold).

**Q: What is a `BroadcastNestedLoopJoin` and when does it appear?**
A: It appears when there is no equi-join condition (e.g., a cross join or non-equi predicate). It's O(M×N) — extremely slow. Always check explain() output for this node and add appropriate join keys if possible.

In [ ]:
# ── FULL OPTIMIZED SOLUTION ───────────────────────────────────────────────────
print("=" * 60)
print("GOOD: BroadcastHashJoin plan")
print("=" * 60)

# Step 1: Read cabs once, project only needed columns, cache
# The cache keeps the 2 MB table in executor memory for reuse
cabs_small = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(CABS_PATH)
    .select(
        F.col("CabNumber"),
        F.col("LicenseType"),
        F.col("VehicleYear").cast(IntegerType()),
        F.col("WheelchairAccessible"),
        F.col("Active")
    )
    .filter(F.col("Active") == "YES")  # push predicate before broadcast
).cache()

# Step 2: Explicitly broadcast the small dimension table
# F.broadcast() injects a BroadcastExchange hint into the logical plan.
# The physical planner MUST honor explicit hints (unlike threshold-based auto-broadcast).
enriched = (
    trips_df
    .join(
        F.broadcast(cabs_small),          # <── the critical hint
        trips_df.cab_number == cabs_small.CabNumber,
        "left"
    )
    .select(
        trips_df.cab_number,
        trips_df.pickup_ts,
        trips_df.fare,
        trips_df.distance,
        trips_df.borough,
        F.col("LicenseType"),
        F.col("VehicleYear"),
        F.col("WheelchairAccessible")
    )
)

# Step 3: Verify plan — must see BroadcastHashJoin, NOT SortMergeJoin
# Must see BroadcastExchange on the cabs side, NO Exchange on the trips side
enriched.explain(mode="formatted")

# Step 4: Action — materialise and show sample
enriched.show(5, truncate=False)

---
## Topic 2 — Shuffle Optimization

### Business Scenario
The TLC finance team runs a **daily aggregation pipeline** that computes total fares, average distance, and trip counts grouped by `LicenseType`, `VehicleYear`, and `borough`. On a 4-core local cluster (or a small production cluster), the default `spark.sql.shuffle.partitions = 200` creates 200 tiny output partitions — each ~100 KB — flooding the task scheduler with overhead and causing excessive small-file writes. Conversely, on a 500-node cluster, 200 partitions might be *too few*, causing each partition to be 200 MB+ with long GC pauses.

### Dataset Description
| Table | Rows | Partitions (bad) | Partitions (good) |
|-------|------|-----------------|-------------------|
| `trips_1m` | 1 000 000 | 200 (default) | 8 (tuned) or AQE coalesced |

### Schema
```
trips_1m: cab_number STRING, license_type STRING, vehicle_year INT,
          borough STRING, fare DOUBLE, distance DOUBLE, trip_date DATE
```

### Why this matters in interviews
Shuffle partition tuning is the **most commonly misunderstood** Spark config. Candidates who can articulate *why* 200 is the wrong default and *how* AQE's `coalescePartitions` fixes it dynamically demonstrate real operational experience.

In [ ]:
# ── Sample Data Generation ────────────────────────────────────────────────────
license_types = ["OWNER MUST DRIVE", "NAMED DRIVER", "AGENT REQUIRED", "CORPORATE"]
vehicle_years = list(range(2010, 2025))
boroughs = ["Manhattan", "Brooklyn", "Queens", "Bronx", "Staten Island"]

trips_1m = (
    spark.range(1_000_000)
    .withColumn("license_type",
        F.element_at(
            F.array([F.lit(l) for l in license_types]),
            (F.col("id") % 4 + 1).cast("int")
        )
    )
    .withColumn("vehicle_year",
        F.element_at(
            F.array([F.lit(y) for y in vehicle_years]),
            (F.col("id") % 15 + 1).cast("int")
        )
    )
    .withColumn("borough",
        F.element_at(
            F.array([F.lit(b) for b in boroughs]),
            (F.col("id") % 5 + 1).cast("int")
        )
    )
    .withColumn("fare",      F.round(F.rand(seed=1) * 80 + 5, 2))
    .withColumn("distance",  F.round(F.rand(seed=2) * 20 + 0.5, 2))
    .withColumn("trip_date", F.date_add(F.lit("2024-01-01"), (F.col("id") % 365).cast("int")))
    .drop("id")
)

# Persist to avoid re-generation in benchmark cells
trips_1m.cache()
row_count = trips_1m.count()
print(f"Generated {row_count:,} trip rows")
trips_1m.printSchema()

### Problem Statement
Compute daily revenue KPIs (total fare, avg distance, trip count) grouped by `license_type` and `vehicle_year`. The poorly optimized version uses `shuffle.partitions=200`, which creates 200 post-shuffle partitions for data that ultimately collapses to ~60 distinct group combinations. This creates 200 tasks writing ~100 KB each — pure scheduler overhead with no parallelism benefit.

In [ ]:
# ── BAD CODE: Default 200 shuffle partitions + unnecessary double-pass ─────────

# Anti-pattern 1: 200 shuffle partitions for ~60 distinct key combinations
spark.conf.set("spark.sql.shuffle.partitions", "200")
# Also disable AQE coalesce so we see the full badness
spark.conf.set("spark.sql.adaptive.coalescePartitions.enabled", "false")

print("BAD: 200 shuffle partitions, AQE coalesce disabled")
print("=" * 60)

# Anti-pattern 2: Two separate groupBy passes (two shuffles) when one would do
# First pass: compute total fares
fare_totals = (
    trips_1m
    .groupBy("license_type", "vehicle_year")
    .agg(F.sum("fare").alias("total_fare"))
)

# Second pass: compute trip counts and avg distance SEPARATELY
trip_stats = (
    trips_1m
    .groupBy("license_type", "vehicle_year")
    .agg(
        F.count("*").alias("trip_count"),
        F.avg("distance").alias("avg_distance")
    )
)

# Anti-pattern 3: Join the two aggregated results — THIRD shuffle!
bad_result = fare_totals.join(trip_stats, ["license_type", "vehicle_year"])

print("Physical plan (notice multiple Exchange nodes):")
bad_result.explain(mode="formatted")

t0 = time.time()
bad_count = bad_result.count()
t_bad = time.time() - t0
print(f"Bad approach: {bad_count} groups in {t_bad:.2f}s")

# Restore for next cell
spark.conf.set("spark.sql.shuffle.partitions", "8")
spark.conf.set("spark.sql.adaptive.coalescePartitions.enabled", "true")

### Interview Questions — Shuffle Optimization

1. Why is `spark.sql.shuffle.partitions = 200` the default and why is it almost always wrong? Derive a formula for the correct partition count given target partition size of 128 MB.

2. Explain the difference between `groupByKey` and `reduceByKey` in the RDD API. Translate that concept to the DataFrame API — what is the DataFrame equivalent of `reduceByKey`?

3. In the bad code above, three shuffles occur. Draw the stage DAG and identify where each `Exchange` node sits. How does combining aggregations into a single `groupBy` eliminate two of those shuffles?

4. What is a `HashAggregate` with a `partial_` prefix in the physical plan, and why does it appear before the Exchange? How does this partial aggregation reduce shuffle data volume?

5. AQE's `coalescePartitions` collapses post-shuffle partitions at runtime. What are the conditions under which AQE will *not* coalesce? (Hint: think about skew detection and min partition count.)

6. You set `shuffle.partitions=8` on a 100-node cluster running a 1 TB shuffle. Each partition is now 125 GB. What happens? Walk through the GC, spill, and OOM failure modes.

7. What is the difference between `spark.sql.shuffle.partitions` and `spark.default.parallelism`? Which one governs DataFrame/SQL shuffles?

8. Your shuffle stage has 200 tasks. 195 finish in 10 seconds, but 5 tasks run for 3 minutes. This is not a data skew problem — partition sizes are uniform. Name three other root causes of straggler tasks in a shuffle stage.

### Expected Logical Plan Discussion

**Bad path logical plan (three separate operations):**
```
Join Inner, (license_type = license_type AND vehicle_year = vehicle_year)
:- Aggregate [license_type, vehicle_year], [sum(fare) AS total_fare]
:  +- Relation trips_1m
+- Aggregate [license_type, vehicle_year], [count(*) AS trip_count, avg(distance)]
   +- Relation trips_1m
```
Catalyst *cannot* merge these three operations even with `CombineAggregates` because they originate from separate DataFrame lineages. Two scans of `trips_1m` + one join = three shuffles minimum.

**Good path logical plan (single groupBy):**
```
Aggregate [license_type, vehicle_year],
  [sum(fare), count(*), avg(distance), max(trip_date)]
+- Relation trips_1m
```
Single aggregate node → Catalyst emits `partial_sum`, `partial_count`, `partial_avg` before the shuffle, then `final_sum`, `final_count`, `final_avg` after. One shuffle, one scan.

**Catalyst rules relevant here:**
- `CombineAggregates` — merges adjacent aggregates when safe
- `PushDownPredicates` — filter `Active = YES` moves below aggregate
- `ReplaceDeduplicateWithAggregate` — dedup patterns become aggregates internally

### Expected Physical Plan Discussion

**Bad path physical plan (3 Exchange nodes):**
```
SortMergeJoin [license_type, vehicle_year]
:- Exchange hashpartitioning(license_type, vehicle_year, 200)   ← Exchange #3
:  +- HashAggregate (partial_sum)
:     +- Exchange hashpartitioning(license_type, vehicle_year, 200)  ← Exchange #1
:        +- HashAggregate (partial_sum)
:           +- InMemoryTableScan [trips_1m]
+- Exchange hashpartitioning(license_type, vehicle_year, 200)   ← Exchange #2
   +- HashAggregate (partial_count, partial_avg)
      +- InMemoryTableScan [trips_1m]
```

**Good path physical plan (1 Exchange node, AQE coalesce):**
```
AdaptiveSparkPlan (isFinalPlan=true)
+- == Final Plan ==
   AQEShuffleRead coalesced                           ← AQE collapses 8→4 partitions
   +- HashAggregate [license_type, vehicle_year] (final_sum, final_count, final_avg)
      +- Exchange hashpartitioning(license_type, vehicle_year, 8)  ← single Exchange
         +- HashAggregate (partial_sum, partial_count, partial_avg)
            +- InMemoryTableScan [trips_1m]
```

**Key node names:**
- `HashAggregate` with `partial_` functions = map-side combiner (pre-shuffle)
- `HashAggregate` with `final_` functions = reduce-side merge (post-shuffle)
- `AQEShuffleRead` with `coalesced` = AQE collapsed empty/small post-shuffle partitions

### Spark UI Investigation Guide — Shuffle Optimization

| UI Location | Metric | What to look for |
|-------------|--------|------------------|
| Stages → Shuffle Write | Bytes written | Total data shuffled; minimize this |
| Stages → Tasks → Duration histogram | Task duration | Bimodal distribution = stragglers |
| SQL plan graph | `Exchange` node count | Each Exchange = one shuffle stage |
| SQL plan graph | `AQEShuffleRead` label | Shows `coalesced` if AQE collapsed partitions |
| Executors → GC Time | JVM GC seconds | > 10% of task time = heap pressure |
| Stages → Spill (Memory) | Bytes spilled | Non-zero = partition too large for executor memory |

**Partition count diagnosis workflow:**
```
1. Run job with AQE enabled
2. SQL tab → click HashAggregate (final) → check output rows
3. AQEShuffleRead → check 'number of output partitions' after coalesce
4. If coalesced from 200 → 4, your static setting of 200 was 50x too high
5. Set shuffle.partitions = (estimated_shuffle_bytes / 128MB), rounded up
```

**Formula for shuffle partition tuning:**
```
target_partitions = ceil(total_shuffle_bytes / target_partition_size_bytes)
# target_partition_size_bytes: 64 MB–200 MB depending on executor memory
# total_shuffle_bytes: read from Spark UI Stages tab after a baseline run
```

In [ ]:
# ── OPTIMIZATION EXERCISE: Fill in the TODOs ─────────────────────────────────
# TODO 1: Set shuffle.partitions to the correct value for ~60 output groups
# TODO 2: Combine all three aggregations into a single groupBy call
# TODO 3: Enable AQE coalesce and verify AQEShuffleRead appears in the plan
# TODO 4: Add a .filter() before the groupBy to push predicate down

# spark.conf.set("spark.sql.shuffle.partitions", "TODO")

# good_result = (
#     trips_1m
#     .filter(TODO)
#     .groupBy(TODO)
#     .agg(
#         TODO
#     )
# )
# good_result.explain(mode="formatted")
# good_result.show(10)

In [ ]:
# ── PERFORMANCE BENCHMARKING: Partition Count Effect ─────────────────────────
results = []

for n_parts in [1, 4, 8, 16, 32, 64, 200]:
    spark.conf.set("spark.sql.shuffle.partitions", str(n_parts))
    spark.conf.set("spark.sql.adaptive.coalescePartitions.enabled", "false")  # disable so we see raw effect

    t0 = time.time()
    (
        trips_1m
        .groupBy("license_type", "vehicle_year")
        .agg(
            F.sum("fare").alias("total_fare"),
            F.count("*").alias("trip_count"),
            F.avg("distance").alias("avg_distance")
        )
        .count()
    )
    elapsed = time.time() - t0
    results.append((n_parts, elapsed))
    print(f"shuffle.partitions={n_parts:>3}  time={elapsed:5.2f}s")

# Restore
spark.conf.set("spark.sql.shuffle.partitions", "8")
spark.conf.set("spark.sql.adaptive.coalescePartitions.enabled", "true")

print("\n--- With AQE coalesce enabled ---")
spark.conf.set("spark.sql.shuffle.partitions", "200")  # start high
t0 = time.time()
(
    trips_1m
    .groupBy("license_type", "vehicle_year")
    .agg(
        F.sum("fare").alias("total_fare"),
        F.count("*").alias("trip_count"),
        F.avg("distance").alias("avg_distance")
    )
    .count()
)
t_aqe = time.time() - t0
print(f"shuffle.partitions=200 + AQE coalesce  time={t_aqe:5.2f}s  (AQE auto-collapses to optimal)")
spark.conf.set("spark.sql.shuffle.partitions", "8")

### Production Best Practices — Shuffle Optimization

1. **Never use 200 as a production shuffle partition count.** Use the formula: `ceil(shuffle_bytes / 128MB)`. AQE coalesce handles over-partitioning but not under-partitioning (it can only reduce, never increase, partitions post-shuffle).

2. **Combine all aggregations into a single `groupBy` call.** Every separate `groupBy` on the same keys is an additional shuffle. Catalyst cannot merge them across DataFrame boundaries.

3. **Push predicates before aggregations.** Filter rows *before* the groupBy to reduce the data shuffled. Catalyst's `PushDownPredicates` rule does this automatically when the filter is on the same DataFrame, but not across join boundaries.

4. **Use `repartition(n, col)` strategically** when you know a large downstream join or groupBy will use that key. Pre-partitioning eliminates the Exchange node for that operation.

5. **Monitor GC time in Spark UI Executors tab.** If GC > 10% of task time, partitions are too large. Either increase partition count or increase executor memory.

6. **Use AQE coalesce as a safety net, not a crutch.** It corrects over-partitioning at runtime but still incurs the overhead of creating 200 empty tasks before collapsing them. The right static setting saves both shuffle write overhead and task scheduling overhead.

### Common Follow-up Questions — Shuffle Optimization

**Q: What is the difference between `repartition()` and `coalesce()`?**
A: `repartition(n)` triggers a full shuffle, guarantees exactly n output partitions with balanced data. `coalesce(n)` avoids a shuffle by merging partitions on the same executor — but can produce skewed partitions and only reduces partition count (cannot increase). Use `repartition` before a join key to pre-partition; use `coalesce` before a write to reduce small files.

**Q: What is `AQEShuffleRead` with `localRead`?**
A: When AQE detects that all shuffle map output for a reduce task is on the same executor, it reads locally (no network transfer). This appears as `AQEShuffleRead localRead` in the plan and is a significant network optimization for non-skewed data.

**Q: Can you tune `spark.sql.shuffle.partitions` mid-job?**
A: Yes — `spark.conf.set()` is session-scoped and takes effect for the next shuffle. This is useful in multi-stage pipelines where different stages have different data volumes.

**Q: What is the `spark.sql.adaptive.coalescePartitions.minPartitionNum` config?**
A: It sets a floor on how many partitions AQE will coalesce down to (default: `spark.default.parallelism`). Prevents AQE from collapsing a 200-partition shuffle down to 1 partition, which would serialize downstream processing.

In [ ]:
# ── FULL OPTIMIZED SOLUTION ───────────────────────────────────────────────────
print("=" * 60)
print("GOOD: Single shuffle, tuned partitions, AQE coalesce")
print("=" * 60)

# Step 1: Configure shuffle partitions
# 60 distinct key combinations → 8 partitions is more than enough on a local cluster
# In prod: ceil(shuffle_bytes_GB * 1024 / 128) — measure from Spark UI baseline run
spark.conf.set("spark.sql.shuffle.partitions", "8")
spark.conf.set("spark.sql.adaptive.coalescePartitions.enabled", "true")
spark.conf.set("spark.sql.adaptive.coalescePartitions.minPartitionNum", "1")

# Step 2: Single groupBy with ALL aggregations combined
# One scan of trips_1m + one shuffle = two stages total
# Catalyst emits partial_(sum|count|sum_for_avg) before the Exchange,
# then final_(sum|count|avg) after — map-side combining reduces shuffle volume
daily_kpis = (
    trips_1m
    .groupBy("license_type", "vehicle_year")  # push groupBy keys first for clarity
    .agg(
        F.sum("fare").alias("total_fare"),
        F.count("*").alias("trip_count"),
        F.avg("distance").alias("avg_distance"),
        F.countDistinct("borough").alias("boroughs_served"),
        F.max("trip_date").alias("last_trip_date")
    )
    .orderBy(F.desc("total_fare"))
)

# Step 3: Show plan — look for:
#   Single Exchange node (one shuffle)
#   HashAggregate with partial_ functions before Exchange
#   AQEShuffleRead coalesced after Exchange
daily_kpis.explain(mode="formatted")

# Step 4: Materialise
t0 = time.time()
daily_kpis.show(20, truncate=False)
print(f"\nCompleted in {time.time()-t0:.2f}s")

---
## Topic 3 — Window Functions

### Business Scenario
The TLC performance team needs to produce a **daily leaderboard** of cabs within each license type: rank cabs by total fares, compute running revenue totals per cab, calculate day-over-day fare growth, and compute a 7-day rolling average. This powers the weekly ops review dashboard.

### Dataset Description
| Table | Rows | Grain | Partition Key |
|-------|------|-------|---------------|
| `cab_daily` | 500 000 | one row per (cab_id, trip_date) | `license_type` |

### Schema
```
cab_daily: cab_id STRING, trip_date DATE, fare DOUBLE,
           license_type STRING, borough STRING, trip_count INT
```

### Why this matters in interviews
Window functions are the most common source of **memory spills and OOM** in production Spark jobs. Candidates who understand RANGE vs ROWS frames, unbounded windows, and how to detect window-related memory pressure in the Spark UI separate themselves from the majority who only know the syntax.

In [ ]:
# ── Sample Data Generation ────────────────────────────────────────────────────
from pyspark.sql.window import Window

license_types_w = ["OWNER MUST DRIVE", "NAMED DRIVER", "AGENT REQUIRED", "CORPORATE"]
boroughs_w      = ["Manhattan", "Brooklyn", "Queens", "Bronx", "Staten Island"]

# Read real cab numbers from Cabs.csv for realistic cab_ids
cabs_raw_w = spark.read.option("header", True).csv(CABS_PATH)
cab_id_list = [r.CabNumber for r in cabs_raw_w.select("CabNumber").limit(100).collect()]

# Generate 500 000 daily cab records
cab_daily = (
    spark.range(500_000)
    .withColumn("cab_id",
        F.element_at(
            F.array([F.lit(c) for c in cab_id_list]),
            (F.col("id") % len(cab_id_list) + 1).cast("int")
        )
    )
    .withColumn("trip_date",
        F.date_add(F.lit("2020-01-01"), (F.col("id") % 1825).cast("int"))  # 5 years
    )
    .withColumn("fare",        F.round(F.rand(seed=99) * 500 + 20, 2))
    .withColumn("trip_count",  (F.rand(seed=11) * 30 + 1).cast(IntegerType()))
    .withColumn("license_type",
        F.element_at(
            F.array([F.lit(l) for l in license_types_w]),
            (F.col("id") % 4 + 1).cast("int")
        )
    )
    .withColumn("borough",
        F.element_at(
            F.array([F.lit(b) for b in boroughs_w]),
            (F.col("id") % 5 + 1).cast("int")
        )
    )
    .drop("id")
)

cab_daily.cache()
print(f"cab_daily rows: {cab_daily.count():,}")
cab_daily.printSchema()

### Problem Statement
Produce per-cab analytics: rank within license_type by total fare, running fare total per cab ordered by date, day-over-day fare growth via lag, and a 7-day rolling average. The bad implementation uses a **self-join** for rank and a **Python UDF loop** for running totals — both trigger unnecessary shuffles and prevent Tungsten code generation.

In [ ]:
# ── BAD CODE: Self-join for rank, Python UDF for running total ────────────────
print("BAD: self-join rank + Python UDF running total")
print("=" * 60)

# Anti-pattern 1: Self-join to compute rank within license_type
# Groups all rows, then for EACH cab counts how many cabs have higher fare — O(N²)
cab_totals_bad = (
    cab_daily
    .groupBy("cab_id", "license_type")
    .agg(F.sum("fare").alias("total_fare"))
)

# Non-equi self-join: for each cab A, count how many cabs B have higher fare in same type
# Falls back to BroadcastNestedLoopJoin or SortMergeJoin with cross-join warning
bad_rank = (
    cab_totals_bad.alias("a")
    .join(
        F.broadcast(cab_totals_bad).alias("b"),
        (F.col("a.license_type") == F.col("b.license_type")) &
        (F.col("a.total_fare") <= F.col("b.total_fare")),
        "left"
    )
    .groupBy("a.cab_id", "a.license_type", "a.total_fare")
    .agg(F.countDistinct("b.cab_id").alias("manual_rank"))
)

# Anti-pattern 2: Python UDF for per-row computation
# Breaks Tungsten whole-stage code generation; serializes JVM→Python→JVM for every row
@F.udf(returnType=DoubleType())
def fare_tax_udf(fare):
    # Simulate a "complex" per-row computation that could be done with native functions
    if fare is None:
        return None
    return round(fare * 0.08625, 2)  # NYC sales tax — trivially replaceable with F.col * 0.08625

bad_with_udf = cab_daily.withColumn("tax_amount", fare_tax_udf(F.col("fare")))

print("Self-join rank plan (notice BroadcastNestedLoopJoin or SortMergeJoin):")
bad_rank.explain(mode="formatted")

### Interview Questions — Window Functions

1. What is the difference between `ROWS BETWEEN` and `RANGE BETWEEN` in a window frame specification? Give a concrete example where they produce different results on a dataset with duplicate `trip_date` values.

2. You define `Window.partitionBy("license_type").orderBy("trip_date")` and call `F.sum("fare").over(w)` with no explicit frame. What is the default frame? Is it ROWS or RANGE? What does Spark use as the upper bound?

3. A window with `rowsBetween(Window.unboundedPreceding, Window.currentRow)` is causing executor OOM on large partitions. Explain the memory lifecycle: what does Spark buffer, when is it released, and what two configs can help?

4. What is the difference between `rank()`, `dense_rank()`, and `row_number()`? For the fare sequence `[100, 100, 90, 80]`, what does each function return for the first two rows?

5. You need a 7-day rolling average of daily fares per cab. Write the window spec. Then explain what changes if the data has missing dates (sparse time series) — should you use ROWS or RANGE?

6. A window function requires a sort within each partition. Walk through the physical plan: what node performs the sort, how many Exchange nodes appear, and under what condition does Spark reuse one sort for multiple window functions on the same spec?

7. What is `ntile(n)` and how would you use it to bucket cabs into fare quartiles within each `license_type`? What happens if the number of rows is not evenly divisible by n?

8. In production you have 4 license types with very uneven row counts — one type has 80% of rows. Your window is partitioned by `license_type`. How does this cause a straggler, and what two strategies can you use to mitigate it without changing the partition key?

### Expected Logical Plan Discussion

**Self-join rank (bad path):**
```
Aggregate [a.cab_id, a.license_type, a.total_fare], [countDistinct(b.cab_id)]
+- Join LeftOuter,
   (a.license_type = b.license_type) AND (a.total_fare <= b.total_fare)
   :- SubqueryAlias a
   :  +- Aggregate [cab_id, license_type] [sum(fare) AS total_fare]
   :     +- Relation cab_daily             ← scan #1
   +- SubqueryAlias b
      +- Aggregate [cab_id, license_type] [sum(fare) AS total_fare]
         +- Relation cab_daily             ← scan #2 (same data, twice!)
```
Two full scans + a non-equi join. The `<=` predicate prevents equi-join optimization — Spark may use `BroadcastNestedLoopJoin` (O(M×N)) or require a cross-join flag for `SortMergeJoin`.

**Window function rank (good path):**
```
Window [rank() windowspecdefinition(license_type, total_fare DESC,
        specifiedwindowframe(RowFrame, unboundedpreceding$(), currentrow$()))]
+- Aggregate [cab_id, license_type] [sum(fare) AS total_fare]
   +- Relation cab_daily                   ← ONE scan
```
One scan + one aggregate + one window = three logical nodes. The `ExtractWindowExpressions` rule separates window columns from the project list and wraps them in a `Window` node.

**Catalyst rules relevant here:**
- `ExtractWindowExpressions` — lifts window functions out of Project into Window node
- `ResolveWindowOrder` — validates ORDER BY present for rank/lead/lag functions
- `EliminateSorts` — removes redundant Sort if data is already partitioned and ordered

### Expected Physical Plan Discussion

**Window function physical plan (good path):**
```
AdaptiveSparkPlan (isFinalPlan=false)
+- Window [rank() windowspecdefinition(license_type, total_fare DESC NULLS LAST,
           specifiedwindowframe(RowFrame, unboundedpreceding$(), currentrow$()))]
   +- Sort [license_type ASC, total_fare DESC NULLS LAST]    ← local sort within partition
      +- Exchange hashpartitioning(license_type, 8)           ← ONE shuffle
         +- HashAggregate (final_sum)
            +- Exchange hashpartitioning(cab_id, license_type, 8)
               +- HashAggregate (partial_sum)
                  +- InMemoryTableScan [cab_daily]
```

**Key observations:**
- The `Window` node buffers an **entire partition** (all rows for one `license_type`) before emitting any output rows — this is where OOM risk lives
- The `Sort` node below Window is local-within-partition, not a global sort — it does NOT trigger a new shuffle stage
- Multiple window functions sharing the same `partitionBy` + `orderBy` + frame are evaluated in **one** `Window` node

**RANGE vs ROWS frame in physical plan output:**
```
-- ROWS frame (explicit — use this for time series):
specifiedwindowframe(RowFrame, unboundedpreceding$(), currentrow$())

-- RANGE frame (Spark default when ORDER BY present, no explicit frame):
specifiedwindowframe(RangeFrame, unboundedpreceding$(), currentrow$())
```
**Critical difference:** With RANGE, all rows that tie on the ORDER BY value are in the "current row" boundary. For a date column with many same-date rows, RANGE and ROWS diverge significantly — RANGE can include more rows than you expect.

### Spark UI Investigation Guide — Window Functions

| UI Location | Metric | What to look for |
|-------------|--------|------------------|
| Stages → Spill (Memory) | Bytes spilled to disk | Non-zero = Window partition exceeds executor on-heap memory |
| Stages → Spill (Disk) | Bytes written to disk | Expensive disk I/O during window sort |
| Tasks → Task Duration | Per-task runtime | One very long task = skewed window partition |
| SQL plan graph | `Window` node metrics | `numInputRows` vs task count — shows partition size |
| Executors → Peak JVM Memory | Bytes | Spike during window stage = large partition buffered in memory |

**Diagnosing window memory spill:**
```
Symptom:   Stages tab shows Spill (Memory) > 0 during window stage
Root cause: Window partition too large for executor on-heap memory
            (unbounded frame buffers the entire partition before output)

Fix options (in order of preference):
  1. Use bounded frames: rowsBetween(-6, 0) instead of unboundedPreceding
  2. Pre-aggregate before window to reduce per-partition row count
  3. Increase spark.sql.shuffle.partitions to create smaller window partitions
  4. Add a secondary key to partitionBy to break large partitions
  5. Increase spark.executor.memory (last resort — masking the root cause)
```

**Check partition size distribution before windowing:**
```python
cab_daily.groupBy("license_type").count().orderBy(F.desc("count")).show()
# If one license_type has 80% of rows → that Window task will straggler
# Fix: repartition on (license_type, cab_id[:1]) to spread load
```

In [ ]:
# ── OPTIMIZATION EXERCISE: Fill in the TODOs ─────────────────────────────────
# TODO 1: Define w_rank — partition by license_type, order by total_fare DESC
# TODO 2: Define w_running — partition by cab_id, order by trip_date,
#         explicit ROWS BETWEEN unboundedPreceding AND currentRow
# TODO 3: Define w_roll7 — partition by cab_id, order by trip_date,
#         rowsBetween(-6, 0) for 7-day rolling window
# TODO 4: Apply rank(), dense_rank(), percent_rank() using w_rank
# TODO 5: Apply running sum, lag(1), and 7-day avg using the other windows
# TODO 6: Replace the Python UDF fare_tax_udf with a native Spark expression

# Hint: Window.partitionBy("x").orderBy("y").rowsBetween(start, end)

# w_rank    = Window.partitionBy(TODO).orderBy(TODO)
# w_running = Window.partitionBy(TODO).orderBy(TODO).rowsBetween(TODO, TODO)
# w_roll7   = Window.partitionBy(TODO).orderBy(TODO).rowsBetween(TODO, TODO)

# cab_agg = cab_daily.groupBy("cab_id", "license_type", "trip_date").agg(
#     F.sum("fare").alias("daily_fare"),
#     F.sum("trip_count").alias("daily_trips")
# )

# result = (
#     cab_agg
#     .withColumn("rank",           F.rank().over(TODO))
#     .withColumn("dense_rank",     F.dense_rank().over(TODO))
#     .withColumn("running_total",  F.sum("daily_fare").over(TODO))
#     .withColumn("prev_day_fare",  F.lag("daily_fare", 1, 0.0).over(TODO))
#     .withColumn("roll7_avg",      F.avg("daily_fare").over(TODO))
#     .withColumn("tax_amount",     F.round(F.col("daily_fare") * 0.08625, 2))  # no UDF!
# )
# result.explain(mode="formatted")
# result.show(10, truncate=False)

In [ ]:
# ── PERFORMANCE BENCHMARKING: Self-join vs Window rank ───────────────────────
import time

cab_totals_bench = (
    cab_daily
    .groupBy("cab_id", "license_type")
    .agg(F.sum("fare").alias("total_fare"))
    .cache()
)
_ = cab_totals_bench.count()

# BAD: self-join rank (using broadcast to avoid timeout on small data)
t0 = time.time()
bad_count = (
    cab_totals_bench.alias("a")
    .join(
        F.broadcast(cab_totals_bench).alias("b"),
        (F.col("a.license_type") == F.col("b.license_type")) &
        (F.col("a.total_fare") <= F.col("b.total_fare")),
        "left"
    )
    .groupBy("a.cab_id", "a.license_type", "a.total_fare")
    .agg(F.countDistinct("b.cab_id").alias("manual_rank"))
    .count()
)
t_bad = time.time() - t0
print(f"Self-join rank:   {t_bad:.2f}s")

# GOOD: window rank
t0 = time.time()
w_rank_bench = Window.partitionBy("license_type").orderBy(F.desc("total_fare"))
good_count = (
    cab_totals_bench
    .withColumn("rank",       F.rank().over(w_rank_bench))
    .withColumn("dense_rank", F.dense_rank().over(w_rank_bench))
    .withColumn("pct_rank",   F.percent_rank().over(w_rank_bench))
    .count()
)
t_good = time.time() - t0
print(f"Window rank:      {t_good:.2f}s")
print(f"Speedup:          {t_bad/t_good:.1f}x  (self-join is O(N²) per partition)")

cab_totals_bench.unpersist()

### Production Best Practices — Window Functions

1. **Always specify the frame explicitly.** The default when `ORDER BY` is present is `RANGE BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW`. This often buffers more data than needed and can cause surprising results with duplicate order values.

2. **Prefer `ROWS` over `RANGE` for time-series.** `RANGE` includes all rows with the same ORDER BY value in the "current row" boundary — on daily data with many same-date rows, running totals will be incorrect.

3. **Use bounded frames for rolling windows.** `rowsBetween(-6, 0)` buffers only 7 rows per output row; `rowsBetween(unboundedPreceding, currentRow)` buffers the entire partition up to the current row. The difference matters at scale.

4. **Combine window functions sharing the same spec.** Multiple `.withColumn()` calls using an identical window spec are collapsed into one `Window` node — Spark buffers the partition only once. Define the spec as a variable and reuse it.

5. **Pre-aggregate before windowing.** If you need per-cab-daily rank, aggregate to (cab_id, date) grain first. Fewer rows per window partition = lower memory pressure.

6. **Never use Python UDFs for row-level math.** Replace `@F.udf` with native DataFrame expressions (`F.col * constant`, `F.round`, `F.when`, `F.regexp_extract`, etc.). Native expressions run inside the JVM with Tungsten codegen — Python UDFs serialize every row across process boundaries.

### Common Follow-up Questions — Window Functions

**Q: Can Spark push predicates (WHERE clauses) below a Window node?**
A: Only if the predicate references input columns, not window output columns. `WHERE rank <= 5` cannot be pushed below the Window node — it must be applied as a post-Window filter. Wrap the window query as a subquery/CTE and filter outside.

**Q: What happens to `lag()` / `lead()` at partition boundaries?**
A: They return `null` for rows where the offset falls outside the partition. Use `lag(col, 1, default_value)` to supply a non-null default. This is critical for day-over-day growth calculations — the first row has no prior day.

**Q: How does `percent_rank()` differ from `rank()`?**
A: `rank()` returns 1-based integers. `percent_rank()` returns `(rank - 1) / (total_rows - 1)` as a double in [0.0, 1.0] — useful for percentile bucketing without upfront knowledge of total count.

**Q: Your rolling window needs to look back exactly 30 calendar days but dates are sparse. ROWS or RANGE?**
A: Use `RANGE BETWEEN INTERVAL 30 DAYS PRECEDING AND CURRENT ROW` with a DATE-typed ORDER BY column. This is a date-range frame (Spark 3.0+). ROWS-based would look back a fixed number of rows regardless of the actual time gap — wrong for sparse time series.

In [ ]:
# ── FULL OPTIMIZED SOLUTION ───────────────────────────────────────────────────
print("=" * 60)
print("GOOD: Window functions — rank, running total, lag, rolling avg")
print("=" * 60)

# Step 1: Pre-aggregate to reduce row count before windowing
cab_agg = (
    cab_daily
    .groupBy("cab_id", "license_type", "trip_date")
    .agg(
        F.sum("fare").alias("daily_fare"),
        F.sum("trip_count").alias("daily_trips")
    )
)

# Step 2: Define window specs — one variable per distinct spec
# Reuse variables: Spark collapses same-spec windows into one physical Window node

# Ranking within license type (no frame needed for rank functions)
w_rank = Window.partitionBy("license_type").orderBy(F.desc("daily_fare"))

# Running total per cab: explicit ROWS frame prevents RANGE-default surprises
w_running = (
    Window
    .partitionBy("cab_id")
    .orderBy("trip_date")
    .rowsBetween(Window.unboundedPreceding, Window.currentRow)  # explicit!
)

# 7-day rolling: bounded frame → Spark does NOT buffer full partition
w_roll7 = (
    Window
    .partitionBy("cab_id")
    .orderBy("trip_date")
    .rowsBetween(-6, 0)   # 7 rows: current + 6 preceding
)

# Step 3: Apply all window functions
# rank, dense_rank, pct_rank share w_rank → ONE Window node in physical plan
# running, lag share w_running → ONE Window node
# roll7 → separate Window node (different frame)
result = (
    cab_agg
    .withColumn("rank_in_type",    F.rank().over(w_rank))
    .withColumn("dense_rank",      F.dense_rank().over(w_rank))
    .withColumn("pct_rank",        F.percent_rank().over(w_rank))
    .withColumn("quartile",        F.ntile(4).over(w_rank))          # fare quartile within type
    .withColumn("running_revenue", F.sum("daily_fare").over(w_running))
    .withColumn("prev_day_fare",   F.lag("daily_fare", 1, 0.0).over(w_running))
    .withColumn("dod_growth_pct",
        F.when(
            F.col("prev_day_fare") > 0,
            F.round((F.col("daily_fare") - F.col("prev_day_fare"))
                    / F.col("prev_day_fare") * 100, 2)
        ).otherwise(F.lit(None).cast(DoubleType()))
    )
    .withColumn("roll7_avg_fare",  F.avg("daily_fare").over(w_roll7))
    # Native expression instead of Python UDF — Tungsten codegen applies
    .withColumn("tax_amount",      F.round(F.col("daily_fare") * 0.08625, 2))
)

# Step 4: Show physical plan
# Look for: multiple Window nodes (one per distinct spec), Sort below each, Exchange before first
result.explain(mode="formatted")

# Step 5: Sample output — top earners per license type with rolling stats
(
    result
    .filter(F.col("rank_in_type") <= 3)
    .orderBy("license_type", "rank_in_type", "trip_date")
    .show(15, truncate=False)
)

---
## Topic 4 — Bucketing

### Business Scenario
The TLC data engineering team runs a **nightly enrichment pipeline** that joins the full trips table (~500 M rows) against the cab registry on `CabNumber`. This join runs every night and is the most expensive step in the pipeline. Without bucketing, every run pays the full shuffle cost — sorting and exchanging the entire trips dataset on the join key.

### Dataset Description
| Table | Rows | Format | Buckets |
|-------|------|--------|---------|
| `trips_bucketed_demo` | 500 000 (demo) | Parquet managed table | 8 buckets on `cab_number` |
| `cabs_bucketed_demo` | 8 970 | Parquet managed table | 8 buckets on `CabNumber` |

### Schema
```
trips: cab_number STRING, fare DOUBLE, distance DOUBLE, pickup_ts TIMESTAMP, borough STRING
cabs:  CabNumber STRING, LicenseType STRING, VehicleYear INT, Active STRING
```

### Why this matters in interviews
Bucketing is a **write-time investment for read-time gain**. It eliminates the Exchange node on repeated joins — but only when both sides have the same bucket count and join key. This nuance (bucket count must match exactly) is the most common interview follow-up and the most common production mistake.

In [ ]:
# ── Sample Data Generation ────────────────────────────────────────────────────
import tempfile, os

BUCKET_DIR = os.path.join(tempfile.gettempdir(), "spark_bucket_demo")
os.makedirs(BUCKET_DIR, exist_ok=True)
TRIPS_PLAIN_PATH = os.path.join(BUCKET_DIR, "trips_plain")
CABS_PLAIN_PATH  = os.path.join(BUCKET_DIR, "cabs_plain")

boroughs_b = ["Manhattan", "Brooklyn", "Queens", "Bronx", "Staten Island"]

cabs_raw_b = (
    spark.read.option("header", True).option("inferSchema", True).csv(CABS_PATH)
    .select(
        F.col("CabNumber"),
        F.col("LicenseType"),
        F.col("VehicleYear").cast(IntegerType()),
        F.col("Active")
    )
    .filter(F.col("Active") == "YES")
)

cab_ids_b = [r.CabNumber for r in cabs_raw_b.select("CabNumber").limit(50).collect()]

trips_raw_b = (
    spark.range(500_000)
    .withColumn("cab_number",
        F.element_at(F.array([F.lit(c) for c in cab_ids_b]),
                     (F.col("id") % 50 + 1).cast("int"))
    )
    .withColumn("fare",     F.round(F.rand(seed=5) * 80 + 5, 2))
    .withColumn("distance", F.round(F.rand(seed=6) * 20 + 0.5, 2))
    .withColumn("pickup_ts",
        F.to_timestamp(F.date_add(F.lit("2024-01-01"), (F.col("id") % 365).cast("int")),
                       "yyyy-MM-dd")
    )
    .withColumn("borough",
        F.element_at(F.array([F.lit(b) for b in boroughs_b]),
                     (F.col("id") % 5 + 1).cast("int"))
    )
    .drop("id")
)

# Write plain Parquet for the bad-path comparison
trips_raw_b.write.mode("overwrite").parquet(TRIPS_PLAIN_PATH)
cabs_raw_b.write.mode("overwrite").parquet(CABS_PLAIN_PATH)

print(f"trips rows: {trips_raw_b.count():,}")
print(f"cabs rows:  {cabs_raw_b.count():,}")
print(f"Plain Parquet written to: {BUCKET_DIR}")

### Problem Statement
Write both tables to disk. Version A: plain Parquet. Version B: bucketed and sorted Parquet via `saveAsTable`. Run the same join against each version, compare `explain()` plans, and measure how the Exchange node disappears when bucket metadata is present and counts match.

In [ ]:
# ── BAD CODE: Plain Parquet — pays shuffle cost on every join run ──────────────
print("BAD: plain Parquet join — Exchange nodes on BOTH sides")
print("=" * 60)

trips_plain = spark.read.parquet(TRIPS_PLAIN_PATH)
cabs_plain  = spark.read.parquet(CABS_PLAIN_PATH)

bad_join_b = trips_plain.join(
    cabs_plain, trips_plain.cab_number == cabs_plain.CabNumber, "inner"
)

# Look for:
#   Exchange hashpartitioning(cab_number, N) on trips side   ← SHUFFLE #1
#   Exchange hashpartitioning(CabNumber, N) on cabs side     ← SHUFFLE #2
#   SortMergeJoin [cab_number], [CabNumber]
bad_join_b.explain(mode="formatted")

t0 = time.time()
bad_cnt = bad_join_b.count()
t_bad = time.time() - t0
print(f"\nPlain join result: {bad_cnt:,} rows in {t_bad:.2f}s")

### Interview Questions — Bucketing

1. Explain the bucketing algorithm: given `bucketBy(8, "cab_number")`, how does Spark determine which bucket a row belongs to? What hash function is used, and is it the same as Java's `hashCode()`?

2. You bucket `trips` on `cab_number` with 8 buckets and `cabs` on `CabNumber` with 4 buckets, then join on `cab_number = CabNumber`. Does the Exchange node disappear? Why or why not?

3. What is `sortBy()` in the context of `bucketBy()` and why is it critical for eliminating the Sort node below SortMergeJoin? When can you safely omit it?

4. Bucketing metadata is stored in the Hive metastore. What happens to bucket metadata when you use `write.parquet(path)` vs `saveAsTable(name)`? When is bucket information silently lost?

5. You have 8 buckets but 500 executor cores available. What is the maximum task parallelism for the bucketed join stage? How do you reconcile bucket count vs core count?

6. A table was bucketed 6 months ago with 8 buckets. The data has grown 10x and you need 64 buckets now. Walk through the migration strategy with minimal downtime on a production pipeline.

7. Delta Lake and Iceberg use Z-ordering and clustering instead of Spark bucketing. How do these differ in terms of Exchange node elimination guarantees?

8. You run `ANALYZE TABLE trips_bucketed COMPUTE STATISTICS FOR COLUMNS cab_number` after writing. How does this interact with AQE's runtime join strategy selection? Can AQE override the planned bucketed join?

### Expected Logical Plan Discussion

**Without bucketing:**
```
Join Inner, (cab_number = CabNumber)
:- Relation parquet [trips_plain]    ← no bucket metadata available
+- Relation parquet [cabs_plain]     ← no bucket metadata available
```
No statistics on hash-partitioning → `EnsureRequirements` rule determines both sides need shuffling → two `Exchange` nodes added during physical planning.

**With bucketing (via `saveAsTable`):**
```
Join Inner, (cab_number = CabNumber)
:- Relation parquet trips_bucketed_demo   ← bucket(8, cab_number) from metastore
+- Relation parquet cabs_bucketed_demo    ← bucket(8, CabNumber)  from metastore
```
Catalyst reads `BucketSpec` from Hive metastore. Both sides satisfy `HashPartitioning(key, 8)` → `EnsureRequirements` does not inject Exchange nodes.

**What Catalyst checks for bucket join elimination:**
1. Matching bucket count on both sides
2. Join key equals the bucket column(s)
3. Table read via `spark.table()` or `spark.read.table()` (metastore path required)
4. `spark.sql.sources.bucketing.enabled = true` (default: true)

**Catalyst rule responsible:** `EnsureRequirements` — the physical planning rule that inserts `Exchange` nodes when required distribution is not already satisfied. With bucketing, the `BucketedScan` already satisfies `HashPartitioning` — no Exchange injected.

### Expected Physical Plan Discussion

**Without bucketing — Exchange on both sides:**
```
AdaptiveSparkPlan (isFinalPlan=false)
+- SortMergeJoin [cab_number], [CabNumber], Inner
   :- Sort [cab_number ASC NULLS FIRST]
   :  +- Exchange hashpartitioning(cab_number, 8)     ← SHUFFLE: trips side
   :     +- FileScan parquet [trips_plain]
   +- Sort [CabNumber ASC NULLS FIRST]
      +- Exchange hashpartitioning(CabNumber, 8)       ← SHUFFLE: cabs side
         +- FileScan parquet [cabs_plain]
```

**With bucketing + sortBy — Exchange AND Sort eliminated:**
```
AdaptiveSparkPlan (isFinalPlan=false)
+- SortMergeJoin [cab_number], [CabNumber], Inner
   :- FileScan parquet trips_bucketed_demo             ← BucketedScan, pre-sorted, no Exchange!
   +- FileScan parquet cabs_bucketed_demo              ← BucketedScan, pre-sorted, no Exchange!
```

**Without sortBy (bucketBy only) — Exchange gone but Sort remains:**
```
+- SortMergeJoin [cab_number], [CabNumber], Inner
   :- Sort [cab_number ASC NULLS FIRST]               ← sort still needed within each bucket
   :  +- FileScan parquet trips_bucketed_nosort
   +- Sort [CabNumber ASC NULLS FIRST]
      +- FileScan parquet cabs_bucketed_nosort
```

**Node to cite in interviews:**
- `FileScan` with `BucketedScan` label = bucketing is active
- Absence of `Exchange` on join inputs = Exchange elimination succeeded
- Absence of `Sort` on join inputs = sortBy() at write time paid off

### Spark UI Investigation Guide — Bucketing

| UI Location | What to check | Signal |
|-------------|--------------|--------|
| SQL plan graph | `Exchange` node on join inputs | Present = bucketing NOT being used |
| SQL plan graph | `FileScan` node label | `BucketedScan` = bucketing active |
| Stages tab | Number of stages for join query | 1 stage (bucketed) vs 3 stages (plain shuffle) |
| Stages → Shuffle Write | Bytes written | 0 B = no shuffle = bucketing working |
| SQL plan graph | `Sort` below join | Present with `bucketBy` but no `sortBy` |

**Confirming bucket metadata via SQL:**
```sql
DESCRIBE EXTENDED trips_bucketed_demo;
-- Look for: Num Buckets = 8, Bucket Columns = [cab_number], Sort Columns = [cab_number]
```

**Silent bucketing failures (most common in production):**
```
1. Bucket count mismatch (8 vs 4)  → Exchange NOT eliminated, no error or warning
2. write.parquet() used            → metadata lost, bucketing silently ignored
3. spark.sql.sources.bucketing.enabled=false → bucketing globally disabled
4. Column name case mismatch       → CabNumber vs cabnumber → not recognized as same key
5. AQE upgrades to BroadcastHashJoin (one side small) → bucket join bypassed silently
```

In [ ]:
# ── OPTIMIZATION EXERCISE: Fill in the TODOs ─────────────────────────────────
# TODO 1: Write trips_raw_b using bucketBy(8, "cab_number").sortBy("cab_number")
#         Use saveAsTable("trips_bucketed_demo") — NOT write.parquet()
# TODO 2: Write cabs_raw_b using bucketBy(8, "CabNumber").sortBy("CabNumber")
#         Use saveAsTable("cabs_bucketed_demo")
# TODO 3: Read both via spark.table() and join on cab_number = CabNumber
# TODO 4: Run explain() — confirm NO Exchange nodes on the join inputs
# TODO 5: Intentionally mismatch bucket counts (8 vs 4) and re-run explain()
#         Observe that Exchange REAPPEARS — this is the silent failure mode

# trips_raw_b.write \
#     .mode("overwrite") \
#     .bucketBy(TODO, TODO) \
#     .sortBy(TODO) \
#     .saveAsTable(TODO)

# cabs_raw_b.write \
#     .mode("overwrite") \
#     .bucketBy(TODO, TODO) \
#     .sortBy(TODO) \
#     .saveAsTable(TODO)

# trips_bkt = spark.table(TODO)
# cabs_bkt  = spark.table(TODO)
# trips_bkt.join(cabs_bkt, trips_bkt.cab_number == cabs_bkt.CabNumber).explain(mode="formatted")

In [ ]:
# ── PERFORMANCE BENCHMARKING: Plain vs Bucketed join ─────────────────────────
import time

spark.conf.set("spark.sql.sources.bucketing.enabled", "true")
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "-1")  # prevent auto-broadcast so we see SMJ

# Write bucketed tables
print("Writing bucketed tables...")
trips_raw_b.write.mode("overwrite").bucketBy(8, "cab_number").sortBy("cab_number") \
    .saveAsTable("trips_bkt_bench")
cabs_raw_b.write.mode("overwrite").bucketBy(8, "CabNumber").sortBy("CabNumber") \
    .saveAsTable("cabs_bkt_bench")
print("Done.")

# Plain join benchmark
t0 = time.time()
plain_cnt = trips_plain.join(cabs_plain, trips_plain.cab_number == cabs_plain.CabNumber).count()
t_plain = time.time() - t0

# Bucketed join benchmark
trips_bkt_b = spark.table("trips_bkt_bench")
cabs_bkt_b  = spark.table("cabs_bkt_bench")

t0 = time.time()
bkt_cnt = trips_bkt_b.join(cabs_bkt_b, trips_bkt_b.cab_number == cabs_bkt_b.CabNumber).count()
t_bkt = time.time() - t0

print(f"\nPlain Parquet join : {t_plain:.2f}s  rows={plain_cnt:,}")
print(f"Bucketed join      : {t_bkt:.2f}s  rows={bkt_cnt:,}")
print(f"Speedup            : {t_plain/t_bkt:.1f}x  (scales dramatically with 500M rows)")

# Confirm bucketed plan
print("\nBucketed join plan:")
trips_bkt_b.join(cabs_bkt_b, trips_bkt_b.cab_number == cabs_bkt_b.CabNumber) \
    .explain(mode="formatted")

spark.sql("DROP TABLE IF EXISTS trips_bkt_bench")
spark.sql("DROP TABLE IF EXISTS cabs_bkt_bench")

### Production Best Practices — Bucketing

1. **Match bucket counts exactly.** If trips has 8 buckets and cabs has 4, Spark cannot eliminate the Exchange. There is no warning — the plan silently falls back to SortMergeJoin with a shuffle. Always validate with `DESCRIBE EXTENDED` after writing.

2. **Choose bucket count as a multiple of your executor core count.** With 8 cores and 8 buckets, each bucket maps to one task — perfect parallelism. With 8 buckets on a 500-core cluster, only 8 tasks can run per stage regardless of core count.

3. **Always use `sortBy()` alongside `bucketBy()`.** Without it, Spark eliminates the Exchange but adds a local Sort before SortMergeJoin. With `sortBy()` matching the join key, even the Sort disappears — files on disk are pre-sorted within each bucket.

4. **Use `saveAsTable()`, not `write.parquet()`.** Bucketing metadata lives in the Hive metastore. Plain path writes discard it — Spark will not use bucketing on the next read. This is the single most common production bucketing mistake.

5. **Bucket on your most-joined column.** Bucketing is a write-time cost amortized across reads. Bucket on the column used in the most frequent, most expensive join in your pipeline.

6. **Validate Exchange elimination every time.** Add `assert "Exchange" not in plan_string` to your pipeline's smoke test. AQE, schema changes, or metastore inconsistencies can silently re-introduce Exchange on a join that previously had none.

### Common Follow-up Questions — Bucketing

**Q: Does bucketing work with Delta Lake tables?**
A: Not in the traditional sense. Delta uses Z-ordering and data skipping, not hash-based bucketing. Z-ordering co-locates data by column value but does not guarantee the deterministic hash-partitioning that Spark's `EnsureRequirements` rule uses to eliminate Exchange nodes. Databricks liquid clustering is the closest Delta equivalent but still lacks the Exchange elimination guarantee.

**Q: Can AQE disable bucketed join optimization at runtime?**
A: Yes — if AQE determines at runtime that one side is small enough for broadcast, it upgrades to BroadcastHashJoin, bypassing the bucketed SortMergeJoin plan entirely. This is usually the right call, but it means your bucket investment is unused for that query.

**Q: What is the maximum number of buckets you should use?**
A: Common guideline: `2 × executor_core_count`. Too many buckets creates a small-file problem (many tiny Parquet files per partition). Too few under-parallelizes the join stage. For a 100-core cluster with 200 MB average bucket size, 200 buckets is a reasonable starting point.

**Q: How do you handle a schema evolution that changes the bucket column?**
A: You must fully rewrite the table. Changing the bucket column or count invalidates all existing bucket files. Strategy: write to a new table name with the new schema, validate with smoke tests, then rename/swap at the metastore level to minimize downtime.

In [ ]:
# ── FULL OPTIMIZED SOLUTION ───────────────────────────────────────────────────
print("=" * 60)
print("GOOD: Bucketed join — Exchange + Sort eliminated, pre-sorted files")
print("=" * 60)

spark.conf.set("spark.sql.sources.bucketing.enabled", "true")
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "-1")

# Step 1: Write with matching bucket count and sort key on BOTH sides
# MUST use saveAsTable — write.parquet() loses bucket metadata!
trips_raw_b.write \
    .mode("overwrite") \
    .bucketBy(8, "cab_number") \
    .sortBy("cab_number") \
    .saveAsTable("trips_bucketed_final")

cabs_raw_b.write \
    .mode("overwrite") \
    .bucketBy(8, "CabNumber") \
    .sortBy("CabNumber") \
    .saveAsTable("cabs_bucketed_final")

# Step 2: Read via spark.table() — required to pick up metastore bucket metadata
# spark.read.parquet(path) would bypass metadata and force shuffle!
trips_final = spark.table("trips_bucketed_final")
cabs_final  = spark.table("cabs_bucketed_final")

# Step 3: Confirm bucket metadata is registered
print("\nTrips bucket metadata:")
spark.sql("DESCRIBE EXTENDED trips_bucketed_final") \
    .filter(F.col("col_name").isin("Num Buckets", "Bucket Columns", "Sort Columns")) \
    .show(truncate=False)

# Step 4: Join — Spark reads bucket N from trips alongside bucket N from cabs,
# matching rows are co-located, no network shuffle required
bucketed_result = (
    trips_final
    .join(cabs_final, trips_final.cab_number == cabs_final.CabNumber, "inner")
    .select(
        trips_final.cab_number,
        trips_final.fare,
        trips_final.pickup_ts,
        trips_final.borough,
        cabs_final.LicenseType,
        cabs_final.VehicleYear
    )
)

# Step 5: Verify plan — MUST see FileScan (BucketedScan) with NO Exchange or Sort
bucketed_result.explain(mode="formatted")

# Step 6: Validate programmatically
plan_str = bucketed_result._jdf.queryExecution().executedPlan().toString()
has_exchange = "Exchange" in plan_str
print(f"\nExchange present in plan: {has_exchange}")
print("PASS: bucket join Exchange elimination confirmed!" if not has_exchange
      else "WARN: Exchange still present — check bucket counts and table read method")

bucketed_result.show(5)

# Cleanup
spark.sql("DROP TABLE IF EXISTS trips_bucketed_final")
spark.sql("DROP TABLE IF EXISTS cabs_bucketed_final")

---
## Topic 5 — Data Skew Detection & Salting

### Business Scenario
The NYC TLC data team discovers that a single medallion cab (`T802127C`, the first row in Cabs.csv) has been running continuously for years and accounts for **80% of all trip records** in the database. Any groupBy or join on `cab_number` creates a catastrophic straggler: 1 task processes 400 000 rows while the other 49 tasks process ~2 000 rows each.

### Dataset Description
| Table | Rows | Distribution | Hot Key |
|-------|------|-------------|---------|
| `skewed_trips` | 500 000 | 80% one cab, 20% others | `T802127C` |
| `cabs_dim` | 8 970 | Uniform | — |

### Schema
```
skewed_trips: cab_number STRING, fare DOUBLE, distance DOUBLE,
              pickup_ts TIMESTAMP, borough STRING
```

### Why this matters in interviews
Data skew is the **number one cause of production Spark job failures** at scale. The salting technique is a standard senior-engineer answer. Interviewers will push on: salt factor selection, small table replication cost, and AQE's automated skew join handling.

In [ ]:
# ── Sample Data Generation: Skewed Distribution ───────────────────────────────
HOT_CAB = "T802127C"   # real cab from Cabs.csv row 1 — the hot key
SALT_FACTOR = 10       # replicate small table 10x, add 0-9 salt to hot key

boroughs_s = ["Manhattan", "Brooklyn", "Queens", "Bronx", "Staten Island"]

# Read all cab numbers for the non-hot pool
cabs_dim = (
    spark.read.option("header", True).option("inferSchema", True).csv(CABS_PATH)
    .select(F.col("CabNumber"), F.col("LicenseType"), F.col("VehicleYear").cast(IntegerType()),
            F.col("Active"))
    .filter(F.col("Active") == "YES")
    .cache()
)
all_cabs = [r.CabNumber for r in cabs_dim.select("CabNumber").collect()]
other_cabs = [c for c in all_cabs if c != HOT_CAB][:49]  # 49 non-hot cabs

# 80% of rows go to HOT_CAB, 20% distributed across other_cabs
hot_trips = (
    spark.range(400_000)   # 80%
    .withColumn("cab_number", F.lit(HOT_CAB))
    .withColumn("fare",     F.round(F.rand(seed=3) * 80 + 5, 2))
    .withColumn("distance", F.round(F.rand(seed=4) * 20 + 0.5, 2))
    .withColumn("pickup_ts",
        F.to_timestamp(F.date_add(F.lit("2024-01-01"), (F.col("id") % 365).cast("int")), "yyyy-MM-dd")
    )
    .withColumn("borough",
        F.element_at(F.array([F.lit(b) for b in boroughs_s]), (F.col("id") % 5 + 1).cast("int"))
    )
    .drop("id")
)

other_trips = (
    spark.range(100_000)   # 20%
    .withColumn("cab_number",
        F.element_at(F.array([F.lit(c) for c in other_cabs]),
                     (F.col("id") % len(other_cabs) + 1).cast("int"))
    )
    .withColumn("fare",     F.round(F.rand(seed=7) * 80 + 5, 2))
    .withColumn("distance", F.round(F.rand(seed=8) * 20 + 0.5, 2))
    .withColumn("pickup_ts",
        F.to_timestamp(F.date_add(F.lit("2024-01-01"), (F.col("id") % 365).cast("int")), "yyyy-MM-dd")
    )
    .withColumn("borough",
        F.element_at(F.array([F.lit(b) for b in boroughs_s]), (F.col("id") % 5 + 1).cast("int"))
    )
    .drop("id")
)

skewed_trips = hot_trips.union(other_trips).cache()
print(f"Total trips: {skewed_trips.count():,}")

# ── SKEW DETECTION: Always run this before any groupBy/join on the key ─────────
print("\nTop 10 cabs by row count (skew profile):")
skewed_trips \
    .groupBy("cab_number") \
    .count() \
    .orderBy(F.desc("count")) \
    .show(10)
# If top row is >> median → skewed. Rule of thumb: top partition > 2x median = investigate

### Problem Statement
Join `skewed_trips` with `cabs_dim` on `cab_number`. Without skew handling, task 0 (processing the `T802127C` partition) handles 400 000 rows while the other 49 tasks handle ~2 000 rows each — an 200x imbalance. The straggler task determines the total stage duration. Solution: **salting** — add a random integer suffix to the hot key and replicate the small table `SALT_FACTOR` times.

In [ ]:
# ── BAD CODE: Regular join on skewed key ──────────────────────────────────────
print("BAD: regular join — T802127C straggler task has 200x more work")
print("=" * 60)

# Disable AQE skew join so we see raw straggler behavior
spark.conf.set("spark.sql.adaptive.skewJoin.enabled", "false")
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "-1")

bad_skew_join = skewed_trips.join(
    cabs_dim, skewed_trips.cab_number == cabs_dim.CabNumber, "left"
)
bad_skew_join.explain(mode="formatted")

t0 = time.time()
bad_skew_count = bad_skew_join.count()
t_bad_skew = time.time() - t0
print(f"Skewed join: {bad_skew_count:,} rows in {t_bad_skew:.2f}s")
print("In Spark UI: check Tasks tab for one task with 400 000 input rows vs others with ~2 000")

# Re-enable for subsequent cells
spark.conf.set("spark.sql.adaptive.skewJoin.enabled", "true")

### Interview Questions — Data Skew Detection & Salting

1. You open the Spark UI and see a shuffle stage where 199 out of 200 tasks finish in 5 seconds but 1 task runs for 8 minutes. Walk through your complete debugging workflow: what tabs do you check, what metrics do you look at, and what queries do you run to confirm data skew?

2. Explain the salting technique step by step. What change do you make to the large table? What change do you make to the small table? Why must the small table be replicated exactly `SALT_FACTOR` times?

3. How do you choose the salt factor? What happens if you over-salt (factor = 1000) on a large table? What happens if you under-salt (factor = 2) on a severely skewed key?

4. AQE introduced automatic skew join handling (`spark.sql.adaptive.skewJoin.enabled`). How does it work, and under what conditions will it NOT detect or fix skew?

5. Salting works for joins. How would you handle skew in a `groupBy` aggregation (not a join) where the hot key has 80% of rows? Describe two different approaches.

6. Your join key is a composite: `(cab_number, borough)`. The skew is only on `cab_number`, not `borough`. Does salting still work? How do you modify the technique?

7. After applying salting, you run the job and the straggler is gone — but the results are wrong. What are three ways salting can introduce incorrect results, and how do you validate correctness?

8. What is the `spark.sql.adaptive.skewJoin.skewedPartitionThresholdInBytes` config? What is its default value and how does AQE use it to detect skewed partitions?

### Expected Logical Plan Discussion

**Bad path (no skew handling):**
```
Join LeftOuter, (cab_number = CabNumber)
:- Relation skewed_trips    ← 80% rows have cab_number = T802127C
+- Relation cabs_dim
```
After the Exchange, partition 0 (the T802127C hash bucket) contains 400 000 rows. All other partitions average 2 000 rows. The join itself is logically correct but physically catastrophic.

**Salted path logical plan:**
```
-- Large table side: add salt column
Project [concat(cab_number, "_", floor(rand() * 10)) AS salted_key, ...]
+- Relation skewed_trips

-- Small table side: explode to SALT_FACTOR copies
Project [concat(CabNumber, "_", exploded_salt) AS salted_key, ...]
+- Explode (array(0,1,2,...,9) AS exploded_salt)
   +- Relation cabs_dim

-- Join on salted key
Join LeftOuter, (salted_key = salted_key)
:- Project [salted_key, ...]   ← trips with salt appended
+- Project [salted_key, ...]   ← cabs replicated 10x with each salt value
```

**Effect on distribution:** T802127C's 400 000 rows are now split across 10 salted keys (`T802127C_0` through `T802127C_9`) — 40 000 rows per key. The straggler partition shrinks 10x.

**Cost:** `cabs_dim` grows from 8 970 rows to 89 700 rows (10x). Broadcast this after salting to avoid any shuffle on the small side.

### Expected Physical Plan Discussion

**Bad path (straggler visible):**
```
AdaptiveSparkPlan (isFinalPlan=true)
+- SortMergeJoin [cab_number], [CabNumber], LeftOuter
   :- Sort [cab_number ASC NULLS FIRST]
   :  +- AQEShuffleRead                         ← partition 0: 400 000 rows (straggler!)
   :     +- Exchange hashpartitioning(cab_number, 8)
   :        +- Union
   :           :- InMemoryTableScan [hot_trips]
   :           +- InMemoryTableScan [other_trips]
   +- Sort [CabNumber ASC NULLS FIRST]
      +- Exchange hashpartitioning(CabNumber, 8)
         +- InMemoryTableScan [cabs_dim]
```

**Salted path:**
```
AdaptiveSparkPlan (isFinalPlan=true)
+- BroadcastHashJoin [salted_key], [salted_key], LeftOuter, BuildRight
   :- Project [concat(cab_number, _, floor(rand() * 10)) AS salted_key, ...]
   :  +- Union [hot_trips, other_trips]
   +- BroadcastExchange HashedRelationBroadcastMode
      +- Project [concat(CabNumber, _, exploded_salt) AS salted_key, ...]
         +- Generate explode(array(0,1,...,9))
            +- InMemoryTableScan [cabs_dim]
```
With salting + broadcast on the expanded small side: **zero shuffles, zero stragglers**. The hot key's 400 000 rows are split across 10 hash buckets of 40 000 rows each — indistinguishable from other keys in task duration.

**AQE Skew Join alternative (automatic):**
```
AdaptiveSparkPlan (isFinalPlan=true)
+- SortMergeJoin [cab_number], [CabNumber], LeftOuter
   :- AQEShuffleRead (isSkewed=true, numPartitions=10)  ← AQE splits the hot partition
   +- AQEShuffleRead (coalesced)
```
AQE's `skewJoin` splits oversized shuffle partitions at runtime without code changes. The plan shows `AQEShuffleRead` with `isSkewed=true` annotation on the split partitions.

### Spark UI Investigation Guide — Data Skew

| UI Location | Metric | Skew Signal |
|-------------|--------|-------------|
| Stages → Tasks | Task Duration column | Most tasks: 5s, one task: 8 min → skew |
| Stages → Tasks | Records Read per task | 1 task shows 200x more records than median |
| Stages → Tasks | Shuffle Read Size | One task reading 200x more bytes from shuffle |
| SQL plan graph | `AQEShuffleRead` | `isSkewed=true` label = AQE detected and split the partition |
| Jobs → Stages | Stage duration | Stage duration = max(task durations), not avg |

**Step-by-step skew debugging workflow:**
```
Step 1: Identify the slow stage
        Jobs tab → find the stage with the longest duration
        Click into the stage

Step 2: Check task duration distribution
        Stages → Tasks → sort by Duration DESC
        If top 1-5 tasks >> median duration → skew

Step 3: Check task input sizes
        Stages → Tasks → look at 'Input Size / Records' column
        Hot partition will show 10-1000x more records

Step 4: Identify the hot key in code
        df.groupBy("join_key").count().orderBy(F.desc("count")).show(10)
        If top row >> 2x average → confirmed skew on that key

Step 5: Choose fix
        Option A: AQE skewJoin (enable and tune threshold) — zero code change
        Option B: Salting — code change, more control, works without AQE
        Option C: Broadcast the small side — if it fits in memory
```

In [ ]:
# ── OPTIMIZATION EXERCISE: Fill in the TODOs ─────────────────────────────────
# TODO 1: Add a salt column to skewed_trips: random integer 0 to SALT_FACTOR-1
#         appended to cab_number → new column "salted_cab_key"
# TODO 2: Expand cabs_dim SALT_FACTOR times using F.explode(F.array(...))
#         Each row gets a different salt value, combined key = CabNumber + "_" + salt
# TODO 3: Join on salted_cab_key (trips) = salted_cab_key (cabs)
# TODO 4: Broadcast the expanded cabs table (still small: 8970 * 10 = 89700 rows)
# TODO 5: Verify task duration is uniform (no straggler) by comparing explain() plans

# salted_trips = skewed_trips.withColumn(
#     "salted_cab_key",
#     F.concat(F.col("cab_number"), F.lit("_"),
#              (F.rand() * TODO).cast(IntegerType()).cast(StringType()))
# )

# cabs_exploded = (
#     cabs_dim
#     .withColumn("salt", F.explode(F.array([F.lit(i) for i in range(TODO)])))
#     .withColumn("salted_cab_key",
#                 F.concat(F.col("CabNumber"), F.lit("_"), F.col("salt").cast(StringType())))
# )

# salted_result = salted_trips.join(
#     F.broadcast(cabs_exploded),
#     "salted_cab_key",
#     "left"
# ).drop("salted_cab_key", "salt")

# salted_result.explain(mode="formatted")
# salted_result.count()

In [ ]:
# ── PERFORMANCE BENCHMARKING: Skewed vs Salted join ──────────────────────────
import time

spark.conf.set("spark.sql.adaptive.skewJoin.enabled", "false")
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "-1")

# BAD: regular join on skewed key
t0 = time.time()
bad_skew_cnt = skewed_trips.join(
    cabs_dim, skewed_trips.cab_number == cabs_dim.CabNumber, "left"
).count()
t_bad = time.time() - t0
print(f"Skewed join (no salt):  {t_bad:.2f}s  rows={bad_skew_cnt:,}")

# GOOD: manual salting + broadcast
# Step 1: Add random salt to trips
salted_trips = skewed_trips.withColumn(
    "salted_key",
    F.concat(
        F.col("cab_number"),
        F.lit("_"),
        (F.rand(seed=42) * SALT_FACTOR).cast(IntegerType()).cast(StringType())
    )
)

# Step 2: Explode cabs_dim SALT_FACTOR times
cabs_exploded = (
    cabs_dim
    .withColumn("salt", F.explode(F.array([F.lit(i) for i in range(SALT_FACTOR)])))
    .withColumn("salted_key",
                F.concat(F.col("CabNumber"), F.lit("_"), F.col("salt").cast(StringType())))
    .drop("salt")
)
print(f"Expanded cabs size: {cabs_exploded.count():,} rows  ({SALT_FACTOR}x original)")

# Step 3: Join on salted key — broadcast the expanded small side
t0 = time.time()
salted_cnt = (
    salted_trips
    .join(F.broadcast(cabs_exploded), "salted_key", "left")
    .drop("salted_key", "CabNumber")
    .count()
)
t_good = time.time() - t0
print(f"Salted join (10x salt): {t_good:.2f}s  rows={salted_cnt:,}")
print(f"Speedup: {t_bad/t_good:.1f}x  (gap widens with larger skew ratio)")

spark.conf.set("spark.sql.adaptive.skewJoin.enabled", "true")

### Production Best Practices — Data Skew

1. **Detect skew before you optimize.** Run `groupBy(join_key).count().orderBy(desc("count")).show(10)` on every new dataset before writing a join. The 5-second diagnosis prevents hours of debugging after the job fails in production.

2. **Try AQE skew join first.** Enable `spark.sql.adaptive.skewJoin.enabled=true` (default in Spark 3.x) and tune `skewedPartitionThresholdInBytes` (default 256 MB). This requires zero code changes and handles moderate skew well.

3. **Use manual salting when AQE is insufficient.** AQE skew detection is threshold-based — it misses skew in small-to-medium datasets where partitions are below the byte threshold even if the ratio is extreme. Manual salting is deterministic and inspectable.

4. **Choose salt factor = number of straggler partitions you want.** A 200x skewed key with 10x salt → 20x max imbalance. With 100x salt → 2x max imbalance but 100x small table replication. Balance: `ceil(hot_partition_rows / avg_partition_rows / 2)`.

5. **Broadcast the salted small table.** After expanding by SALT_FACTOR, if the small table still fits in broadcast threshold (e.g., 8970 × 10 = 89700 rows ≈ 20 MB), broadcast it. This eliminates the shuffle on both sides entirely.

6. **Validate row counts before and after salting.** The result of a salted join must have exactly the same row count as an unoptimized join on a small sample. Add a unit test that compares counts on a 1% sample.

7. **Document hot keys.** In production, hot keys change over time. Add a data quality check that alerts when any single key exceeds 10% of total rows — catch skew before it becomes a production incident.

### Common Follow-up Questions — Data Skew

**Q: AQE skew join splits the hot partition. Doesn't this violate sort order for SortMergeJoin?**
A: AQE is aware of sort requirements. When it splits a skewed partition, it also reads the corresponding range from the other side to maintain join correctness. The matching rows on the right side are duplicated across the split tasks (this is the "skew reader" pattern).

**Q: Your groupBy on a skewed key is slow — salting doesn't apply to aggregations. What do you do?**
A: Two approaches: (1) Two-phase aggregation — add a random salt column, groupBy `(key, salt)` for partial sums, then groupBy `key` to merge. (2) `repartition(n, key)` with high n to spread the hot key across multiple partitions before the aggregate.

**Q: Salting works for left joins. What about right joins and full outer joins?**
A: For right joins, salt the right table (the large side) and replicate the left (small side). For full outer joins, salting is complex — unmatched rows from both sides need to be correctly attributed. In practice, split full outer joins into left join + anti-join, salt separately.

**Q: What is `spark.sql.adaptive.skewJoin.skewedPartitionFactor`?**
A: A multiplier (default 5) applied to the median partition size. A partition is considered skewed if its size > `skewedPartitionFactor × median size` AND `> skewedPartitionThresholdInBytes`. Both conditions must hold.

In [ ]:
# ── FULL OPTIMIZED SOLUTION ───────────────────────────────────────────────────
print("=" * 60)
print("GOOD: Salted join + broadcast — no straggler, zero shuffles")
print("=" * 60)

spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "-1")  # prove it works without auto-broadcast

# ── Step 1: Detect skew (always do this first in production) ──────────────────
print("\nSkew profile before optimization:")
skewed_trips.groupBy("cab_number").count() \
    .orderBy(F.desc("count")).show(5, truncate=False)

total = skewed_trips.count()
hot_count = skewed_trips.filter(F.col("cab_number") == HOT_CAB).count()
print(f"Hot key {HOT_CAB!r} accounts for {hot_count/total*100:.1f}% of rows → SEVERE SKEW")

# ── Step 2: Salt the large table ──────────────────────────────────────────────
# Append a random integer 0 to SALT_FACTOR-1 to the join key
# This distributes the hot key across SALT_FACTOR partitions
trips_salted = skewed_trips.withColumn(
    "salted_key",
    F.concat(
        F.col("cab_number"),
        F.lit("_"),
        (F.rand(seed=42) * SALT_FACTOR).cast(IntegerType()).cast(StringType())
    )
)

# ── Step 3: Replicate the small table SALT_FACTOR times ───────────────────────
# Every original key T802127C must appear as T802127C_0, T802127C_1, ..., T802127C_9
# so that every salted trip row has a matching cabs row
cabs_salted = (
    cabs_dim
    .withColumn("salt_val", F.explode(F.array([F.lit(i) for i in range(SALT_FACTOR)])))
    .withColumn("salted_key",
                F.concat(F.col("CabNumber"), F.lit("_"),
                         F.col("salt_val").cast(StringType())))
    .drop("salt_val", "CabNumber")   # drop original key to avoid ambiguity
)

print(f"\nExpanded cabs: {cabs_salted.count():,} rows = {SALT_FACTOR}x original")
print(f"Still fits in broadcast: ~{SALT_FACTOR * 9000 / 1e6:.1f} MB")

# ── Step 4: Join on salted key — broadcast the expanded small side ─────────────
# BroadcastHashJoin on ~90K rows → zero shuffles, zero stragglers
result_salted = (
    trips_salted
    .join(F.broadcast(cabs_salted), "salted_key", "left")
    .drop("salted_key")   # drop the synthetic salt column from output
)

# ── Step 5: Verify plan ───────────────────────────────────────────────────────
# EXPECTED: BroadcastHashJoin, Generate (explode), NO Exchange on trips side
result_salted.explain(mode="formatted")

# ── Step 6: Benchmark and validate correctness ────────────────────────────────
t0 = time.time()
result_count = result_salted.count()
t_salted = time.time() - t0
print(f"\nSalted result: {result_count:,} rows in {t_salted:.2f}s")
print(f"Original row count: {total:,} — counts match: {result_count == total}")

result_salted.show(5, truncate=False)

---
## Topic 6 — Adaptive Query Execution (AQE)

### Business Scenario
The TLC end-of-day reporting pipeline runs three queries: (1) a fare aggregation by license type, (2) a join of trips against cab registry, and (3) a skewed aggregation of hot cabs. Historically these were individually tuned with static configs. AQE allows the engine to dynamically re-plan each query at runtime — adjusting partition counts, switching join strategies, and splitting skewed partitions — without code changes.

### Dataset Description
Reuses `trips_1m`, `cabs_dim`, and `skewed_trips` from earlier topics.

### AQE Configuration Reference Table
| Config Key | Default | Recommended (Prod) | What it does |
|-----------|---------|-------------------|-------------|
| `spark.sql.adaptive.enabled` | `true` (Spark 3.2+) | `true` | Master switch for AQE |
| `spark.sql.adaptive.coalescePartitions.enabled` | `true` | `true` | Merge small post-shuffle partitions |
| `spark.sql.adaptive.coalescePartitions.minPartitionNum` | `spark.default.parallelism` | `1` (let AQE decide) | Floor on coalesced partition count |
| `spark.sql.adaptive.advisoryPartitionSizeInBytes` | `64MB` | `128MB–256MB` | Target partition size after coalesce |
| `spark.sql.adaptive.skewJoin.enabled` | `true` | `true` | Auto-split skewed shuffle partitions |
| `spark.sql.adaptive.skewJoin.skewedPartitionThresholdInBytes` | `256MB` | `64MB–128MB` (aggressive) | Min size to classify as skewed |
| `spark.sql.adaptive.autoBroadcastJoinThreshold` | same as static | `10MB–50MB` | Runtime broadcast upgrade threshold |

### Why this matters in interviews
AQE is a **mandatory topic** at the Staff+ level since Spark 3.0. Candidates must explain all three features (coalesce, join strategy switching, skew join), cite the relevant plan nodes (`AdaptiveSparkPlan`, `AQEShuffleRead`), and — critically — know when to **disable** AQE.

In [ ]:
# ── Sample Data Setup: Reuse earlier datasets ─────────────────────────────────
# trips_1m, cabs_dim, skewed_trips are already cached from earlier topics.
# Verify they are still in memory:
print("Checking cached DataFrames:")
print(f"  trips_1m     : {trips_1m.count():,} rows (cached={trips_1m.is_cached})")
print(f"  cabs_dim     : {cabs_dim.count():,} rows (cached={cabs_dim.is_cached})")
print(f"  skewed_trips : {skewed_trips.count():,} rows (cached={skewed_trips.is_cached})")
print()
print("Current AQE settings:")
for key in [
    "spark.sql.adaptive.enabled",
    "spark.sql.adaptive.coalescePartitions.enabled",
    "spark.sql.adaptive.skewJoin.enabled",
    "spark.sql.shuffle.partitions",
]:
    print(f"  {key} = {spark.conf.get(key)}")

### Problem Statement
Demonstrate all three AQE features on the TLC dataset:
1. **Coalesce partitions** — run aggregation with 200 shuffle partitions, AQE collapses empty/small ones
2. **Join strategy switching** — set up a SortMergeJoin that AQE upgrades to BroadcastHashJoin at runtime
3. **Skew join** — join on `skewed_trips`, AQE auto-splits the hot partition

The bad code disables AQE entirely, forcing static 200-partition shuffles, no runtime broadcast upgrades, and no skew handling.

In [ ]:
# ── BAD CODE: AQE disabled — static 200 partitions, no runtime optimization ───
print("BAD: AQE disabled — 200 partitions, no broadcast upgrade, no skew handling")
print("=" * 70)

spark.conf.set("spark.sql.adaptive.enabled",                      "false")
spark.conf.set("spark.sql.shuffle.partitions",                    "200")
spark.conf.set("spark.sql.autoBroadcastJoinThreshold",            "-1")

# Feature 1 (missing): No coalesce — 200 tiny partitions written to disk
q1 = (
    trips_1m
    .groupBy("license_type", "vehicle_year")
    .agg(F.sum("fare").alias("total_fare"), F.count("*").alias("trips"))
)
print("\nQ1 — Aggregation plan (AQE OFF, 200 partitions, no coalesce):")
q1.explain(mode="formatted")

# Feature 2 (missing): No runtime broadcast upgrade — forced SortMergeJoin
# even though cabs_dim is tiny (8970 rows ~2 MB)
q2 = trips_1m.join(cabs_dim, trips_1m.license_type == cabs_dim.LicenseType, "left")
print("\nQ2 — Join plan (AQE OFF, no broadcast upgrade, forced SortMergeJoin):")
q2.explain(mode="formatted")

t0 = time.time()
cnt1, cnt2 = q1.count(), q2.count()
t_bad_aqe = time.time() - t0
print(f"\nAQE OFF total time: {t_bad_aqe:.2f}s  (q1={cnt1}, q2={cnt2:,})")

# Restore AQE
spark.conf.set("spark.sql.adaptive.enabled", "true")

### Interview Questions — AQE

1. AQE has three main features in Spark 3.x. Name all three, explain the mechanism of each, and give a concrete example of when each one fires.

2. What is an `AdaptiveSparkPlan` node in the physical plan? When does it transition from `isFinalPlan=false` to `isFinalPlan=true`? What triggers that transition?

3. AQE coalesces post-shuffle partitions. The initial plan has 200 partitions. After the shuffle, AQE collapses to 4. What is the exact mechanism: which node is added to the plan, and what information does AQE use to decide how many partitions to merge?

4. AQE can upgrade a `SortMergeJoin` to `BroadcastHashJoin` at runtime. What is the key difference between the static `autoBroadcastJoinThreshold` and the AQE-specific `spark.sql.adaptive.autoBroadcastJoinThreshold`? Can one be set independently of the other?

5. **When would you disable AQE?** Name at least three specific scenarios where AQE causes correctness issues, performance regression, or operational problems.

6. AQE's skew join detection uses two thresholds: `skewedPartitionThresholdInBytes` and `skewedPartitionFactor`. Explain how both thresholds interact and why both conditions must be true for a partition to be classified as skewed.

7. You have a complex multi-stage pipeline with 20 Spark queries in sequence. AQE is enabled. A downstream query re-plans because AQE changed the output partition count of an upstream query. How can this cascade of re-planning affect determinism and observability?

8. AQE's `AdaptiveSparkPlan` wraps the entire query plan. How does this interact with `explain()` output? Why does `explain()` before execution show a different plan than `explain()` after execution?

### Expected Logical Plan Discussion

**AQE does not modify the logical plan.** All three AQE features are physical-plan-level optimizations applied at runtime. The logical plan is fixed after the analysis and optimization phases.

**What AQE changes:**
- At each shuffle boundary, AQE materializes shuffle map output statistics
- It then re-evaluates physical planning rules using actual byte counts rather than estimates
- The plan is re-compiled in the `AdaptiveSparkPlan` framework and the new physical plan is substituted

**Catalyst rules that AQE re-triggers at runtime:**
- `JoinSelection` — can now upgrade SMJ to BHJ because actual right-side size is known
- `CoalesceShufflePartitions` — merges post-shuffle partitions based on actual shuffle output sizes
- `OptimizeSkewedJoin` — detects skewed partitions from shuffle map statistics and splits them

**Logical plan before AQE (static):**
```
Aggregate [license_type, vehicle_year] [sum(fare), count(*)]
+- Relation trips_1m
```
This is unchanged by AQE. The physical plan below it changes at each materialization point.

### Expected Physical Plan Discussion

**AQE Feature 1 — Coalesce Partitions:**
```
AdaptiveSparkPlan (isFinalPlan=true)
+- == Final Plan ==
   AQEShuffleRead coalesced (numPartitions=4)     ← collapsed 200 → 4
   +- HashAggregate [license_type, vehicle_year] (final_sum, final_count)
      +- Exchange hashpartitioning(license_type, vehicle_year, 200)
         +- HashAggregate (partial_sum, partial_count)
            +- InMemoryTableScan
```
`AQEShuffleRead coalesced` is the new physical node inserted between the Exchange output and the final HashAggregate. It reads shuffle map files and merges adjacent small partitions until `advisoryPartitionSizeInBytes` is reached.

**AQE Feature 2 — Join Strategy Switching:**
```
AdaptiveSparkPlan (isFinalPlan=true)
+- == Final Plan ==
   BroadcastHashJoin [license_type], [LicenseType], LeftOuter, BuildRight
   :- InMemoryTableScan [trips_1m]
   +- BroadcastExchange HashedRelationBroadcastMode(List(LicenseType))
      +- InMemoryTableScan [cabs_dim]
```
The plan originally showed `SortMergeJoin`. After the shuffle on the right side completes, AQE observes the actual byte count (~2 MB for cabs_dim) is below `autoBroadcastJoinThreshold`. It re-plans the join as `BroadcastHashJoin`.

**AQE Feature 3 — Skew Join:**
```
AdaptiveSparkPlan (isFinalPlan=true)
+- SortMergeJoin [cab_number], [CabNumber], LeftOuter
   :- AQEShuffleRead (skewed partitions split into 10)
   :  +- Exchange hashpartitioning(cab_number, 8)
   +- AQEShuffleRead (corresponding right side duplicated)
      +- Exchange hashpartitioning(CabNumber, 8)
```
`AQEShuffleRead` annotated with skew information — the original partition 0 (T802127C, 400K rows) is split into 10 sub-partitions of ~40K rows each. The corresponding right-side partition is read 10 times (once per sub-partition).

**Key node names:**
- `AdaptiveSparkPlan` — root wrapper for entire AQE-managed plan
- `AQEShuffleRead` — replaces plain shuffle read; shows `coalesced`, `localRead`, or skew info
- `isFinalPlan=false` → plan may still change; `isFinalPlan=true` → execution complete

### Spark UI Investigation Guide — AQE

| UI Location | What to check | What AQE shows |
|-------------|--------------|----------------|
| SQL plan graph | Root node | `AdaptiveSparkPlan` wraps all AQE-managed queries |
| SQL plan graph | Shuffle read nodes | `AQEShuffleRead` replaces plain shuffle read |
| SQL plan graph | `AQEShuffleRead` label | `coalesced`, `localRead`, or skew split info |
| SQL plan graph | Join node | `BroadcastHashJoin` where static plan said `SortMergeJoin` = AQE upgraded |
| Stages → Shuffle Write | Final stage count | Fewer stages = AQE coalesced → fewer downstream tasks |
| SQL plan graph | `isFinalPlan` attribute | `false` = pre-execution estimate; `true` = actual executed plan |

**Reading the AQE plan in `explain()` output:**
```
== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false       ← before execution: initial estimate
+- SortMergeJoin [...]                    ← planned strategy (may change)

-- After execution (via df.explain() called after an action):
AdaptiveSparkPlan isFinalPlan=true
+- == Final Plan ==
   BroadcastHashJoin [...]                ← actual strategy used (AQE upgraded)
+- == Initial Plan ==
   SortMergeJoin [...]                    ← what was planned before execution
```

**Checking AQE coalesce outcome:**
```python
# After running an aggregation with AQE enabled:
df_result = trips_1m.groupBy("license_type", "vehicle_year").agg(F.sum("fare"))
df_result.collect()  # trigger action
# Then call explain() to see isFinalPlan=true and AQEShuffleRead coalesced
df_result.explain(mode="formatted")
```

In [ ]:
# ── OPTIMIZATION EXERCISE: Fill in the TODOs ─────────────────────────────────
# TODO 1: Enable all AQE features with production-grade settings
# TODO 2: Set shuffle.partitions=200 (deliberately high — let AQE coalesce)
# TODO 3: Run the aggregation and observe AQEShuffleRead coalesced in the plan
# TODO 4: Join trips_1m with cabs_dim WITHOUT an explicit broadcast hint
#         Observe that AQE upgrades to BroadcastHashJoin at runtime
# TODO 5: Join skewed_trips with cabs_dim and observe AQE skew join behavior
# TODO 6: Call explain() AFTER the action to see isFinalPlan=true

# spark.conf.set("spark.sql.adaptive.enabled",                       "TODO")
# spark.conf.set("spark.sql.adaptive.coalescePartitions.enabled",    "TODO")
# spark.conf.set("spark.sql.adaptive.skewJoin.enabled",              "TODO")
# spark.conf.set("spark.sql.adaptive.advisoryPartitionSizeInBytes",  "TODO")
# spark.conf.set("spark.sql.shuffle.partitions",                     "TODO")

# # AQE Feature 1: Coalesce
# q1 = trips_1m.groupBy("license_type", "vehicle_year").agg(F.sum("fare"))
# q1.count()  # trigger action
# q1.explain(mode="formatted")  # call AFTER action → isFinalPlan=true, AQEShuffleRead coalesced

# # AQE Feature 2: Join strategy switching (no explicit broadcast hint)
# q2 = trips_1m.join(cabs_dim, "TODO", "left")
# q2.count()
# q2.explain(mode="formatted")  # should show BroadcastHashJoin in final plan

# # AQE Feature 3: Skew join
# q3 = skewed_trips.join(cabs_dim, skewed_trips.cab_number == cabs_dim.CabNumber, "left")
# q3.count()
# q3.explain(mode="formatted")  # should show AQEShuffleRead with skew annotations

In [ ]:
# ── PERFORMANCE BENCHMARKING: AQE OFF vs AQE ON ──────────────────────────────
import time

queries = {
    "agg":   lambda: trips_1m.groupBy("license_type","vehicle_year").agg(F.sum("fare")).count(),
    "join":  lambda: trips_1m.join(cabs_dim, trips_1m.license_type == cabs_dim.LicenseType).count(),
    "skew":  lambda: skewed_trips.join(cabs_dim, skewed_trips.cab_number == cabs_dim.CabNumber).count(),
}

print(f"{'Query':<8} {'AQE OFF':>10} {'AQE ON':>10} {'Speedup':>9}")
print("-" * 42)

for name, fn in queries.items():
    # OFF
    spark.conf.set("spark.sql.adaptive.enabled", "false")
    spark.conf.set("spark.sql.shuffle.partitions", "200")
    spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "-1")
    t0 = time.time(); fn(); t_off = time.time() - t0

    # ON
    spark.conf.set("spark.sql.adaptive.enabled", "true")
    spark.conf.set("spark.sql.adaptive.coalescePartitions.enabled", "true")
    spark.conf.set("spark.sql.adaptive.skewJoin.enabled", "true")
    spark.conf.set("spark.sql.adaptive.autoBroadcastJoinThreshold", "50mb")
    spark.conf.set("spark.sql.shuffle.partitions", "200")
    t0 = time.time(); fn(); t_on = time.time() - t0

    print(f"{name:<8} {t_off:>9.2f}s {t_on:>9.2f}s {t_off/t_on:>8.1f}x")

# Restore
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "-1")
spark.conf.set("spark.sql.shuffle.partitions", "8")

### Production Best Practices — AQE

1. **Keep AQE enabled in production (Spark 3.2+).** The default is `true` for good reason. The overhead of shuffle map statistics collection is minimal compared to the gains from partition coalesce and join strategy switching.

2. **Set `advisoryPartitionSizeInBytes` to 128–256 MB** instead of the 64 MB default. Larger partitions reduce task overhead and align with typical HDFS block sizes. Tune per cluster based on executor memory.

3. **Lower `skewedPartitionThresholdInBytes` to 64–128 MB** in skew-prone pipelines. The 256 MB default means AQE won't split a 200 MB skewed partition — which is still 3x the average. Aggressive detection catches more skew.

4. **Do not rely on AQE as a substitute for understanding your data.** AQE handles symptoms; understanding your data's distribution (run the groupBy count profile) fixes root causes. AQE's skew detection can miss cases where partitions are below the byte threshold but are still CPU-intensive.

5. **Disable AQE for deterministic unit tests.** AQE's runtime re-planning means two identical-looking queries may produce different physical plans depending on data volume. In tests that assert specific plan structure, disable AQE to get consistent `explain()` output.

6. **When to disable AQE in production:**
   - When using **bucketed joins** — AQE can inadvertently upgrade to BroadcastHashJoin, bypassing your carefully planned bucket join and re-introducing a shuffle
   - When debugging with `explain()` — AQE's `isFinalPlan=false` initial plan is misleading; disable to see the true static plan
   - When **deterministic partition counts** are required downstream (e.g., writing exactly N files per partition to an external system)

### Common Follow-up Questions — AQE

**Q: When would you DISABLE AQE? Any edge cases where it hurts performance?**
A: Three concrete cases: (1) **Bucketed join bypass** — if AQE's runtime broadcast upgrade fires on a bucketed table join, it re-introduces a shuffle that your bucketing was designed to eliminate. Fix: set `spark.sql.adaptive.autoBroadcastJoinThreshold` lower than the bucketed table size. (2) **Over-coalescing** — if AQE collapses 200 shuffle partitions into 2 but you have 200 executor cores, downstream tasks run serially. Fix: set `coalescePartitions.minPartitionNum = num_cores`. (3) **Plan instability in tests** — AQE produces different final plans for different data volumes; unit tests asserting plan structure become non-deterministic. Disable AQE in test environments.

**Q: Does AQE work with Structured Streaming?**
A: Partially. AQE is supported in streaming with the `stateful` operator but with restrictions. Coalesce is not applied to streaming shuffle outputs. Join strategy switching can fire for stream-static joins. Skew join is not supported in streaming as of Spark 3.3.

**Q: What is `spark.sql.adaptive.forceApply`?**
A: Forces AQE to wrap all queries in `AdaptiveSparkPlan` even when AQE optimizations would not normally apply (e.g., no shuffle in the plan). Useful for testing but adds overhead in production.

**Q: AQE re-plans at shuffle boundaries. What about non-shuffle operations like `mapPartitions`?**
A: AQE only re-plans at shuffle and broadcast exchange boundaries — these are the materialization points where actual byte statistics become available. Pure CPU-bound transformations like `mapPartitions` do not provide statistics and do not trigger AQE re-planning.

In [ ]:
# ── FULL OPTIMIZED SOLUTION: All 3 AQE features demonstrated ─────────────────
print("=" * 70)
print("GOOD: AQE enabled — demonstrating all 3 features")
print("=" * 70)

# ── Configure AQE with production-grade settings ──────────────────────────────
spark.conf.set("spark.sql.adaptive.enabled",                               "true")
spark.conf.set("spark.sql.adaptive.coalescePartitions.enabled",            "true")
spark.conf.set("spark.sql.adaptive.coalescePartitions.minPartitionNum",    "1")
spark.conf.set("spark.sql.adaptive.advisoryPartitionSizeInBytes",          "134217728")  # 128 MB
spark.conf.set("spark.sql.adaptive.skewJoin.enabled",                      "true")
spark.conf.set("spark.sql.adaptive.skewJoin.skewedPartitionThresholdInBytes", "67108864")  # 64 MB
spark.conf.set("spark.sql.adaptive.autoBroadcastJoinThreshold",            "52428800")   # 50 MB
spark.conf.set("spark.sql.shuffle.partitions",                             "200")  # start high, AQE will coalesce

print("\nAQE configuration:")
aqe_configs = [
    "spark.sql.adaptive.enabled",
    "spark.sql.adaptive.coalescePartitions.enabled",
    "spark.sql.adaptive.advisoryPartitionSizeInBytes",
    "spark.sql.adaptive.skewJoin.enabled",
    "spark.sql.adaptive.skewJoin.skewedPartitionThresholdInBytes",
    "spark.sql.adaptive.autoBroadcastJoinThreshold",
    "spark.sql.shuffle.partitions",
]
for k in aqe_configs:
    print(f"  {k} = {spark.conf.get(k)}")

# ── Feature 1: Coalesce partitions ────────────────────────────────────────────
print("\n--- AQE Feature 1: Coalesce Partitions ---")
# Start with 200 shuffle partitions, AQE coalesces to ~4 (60 distinct groups)
q_agg = (
    trips_1m
    .groupBy("license_type", "vehicle_year")
    .agg(
        F.sum("fare").alias("total_fare"),
        F.count("*").alias("trips")
    )
)
t0 = time.time()
agg_count = q_agg.count()
t_agg = time.time() - t0
# Call explain() AFTER the action → isFinalPlan=true, shows actual AQEShuffleRead coalesced
q_agg.explain(mode="formatted")
print(f"Result: {agg_count} groups in {t_agg:.2f}s")
print("Look for AQEShuffleRead with 'coalesced' in the final plan above")

# ── Feature 2: Join strategy switching ───────────────────────────────────────
print("\n--- AQE Feature 2: Join Strategy Switching ---")
# No explicit broadcast hint — AQE will upgrade to BroadcastHashJoin at runtime
# because cabs_dim (~2 MB) is below autoBroadcastJoinThreshold (50 MB)
q_join = trips_1m.join(
    cabs_dim,
    trips_1m.license_type == cabs_dim.LicenseType,
    "left"
)
# Before action: explain() shows AdaptiveSparkPlan isFinalPlan=false with SortMergeJoin
print("Plan BEFORE execution (isFinalPlan=false):")
q_join.explain(mode="formatted")

t0 = time.time()
join_count = q_join.count()
t_join = time.time() - t0

# After action: plan shows isFinalPlan=true with BroadcastHashJoin
print(f"\nPlan AFTER execution ({join_count:,} rows, {t_join:.2f}s):")
q_join.explain(mode="formatted")
print("Look for BroadcastHashJoin in final plan — AQE upgraded from SortMergeJoin")

# ── Feature 3: Skew join handling ─────────────────────────────────────────────
print("\n--- AQE Feature 3: Skew Join ---")
# skewed_trips has 80% rows on T802127C — AQE should detect and split
spark.conf.set("spark.sql.adaptive.skewJoin.skewedPartitionThresholdInBytes", "1048576")  # 1MB to trigger on small data
spark.conf.set("spark.sql.adaptive.skewJoin.skewedPartitionFactor",           "2")

q_skew = skewed_trips.join(
    cabs_dim,
    skewed_trips.cab_number == cabs_dim.CabNumber,
    "left"
)
t0 = time.time()
skew_count = q_skew.count()
t_skew = time.time() - t0
q_skew.explain(mode="formatted")
print(f"Skew join: {skew_count:,} rows in {t_skew:.2f}s")
print("Look for AQEShuffleRead with skew annotations in the plan above")

# ── Final cleanup ─────────────────────────────────────────────────────────────
for df in [cabs_df, cabs_small, trips_1m, cab_daily, cabs_dim, skewed_trips]:
    try:
        df.unpersist()
    except Exception:
        pass
print("\nAll cached DataFrames released. Notebook complete.")